In [ ]:
!pip install flask requests python-dotenv python-docx unidecode
# KB support
import io
try:
    import docx  # python-docx
except Exception:
    # if missing, tell the user clearly
    raise RuntimeError("python-docx is required: pip install python-docx")


In [ ]:
# ===========================================
# PlayStayTion Voice Agent (Telnyx AI Assistant)
# - Word .docx Knowledge Base injection
# - Google Sheets write via Apps Script webhook
# - Cloudflared quick tunnel (no account needed)
# - Same marker-based system prompt
# ===========================================

import os, json, time, hmac, hashlib, threading, re, subprocess, sys, stat, pathlib, requests
from flask import Flask, request, jsonify
from dotenv import load_dotenv
from docx import Document
from unidecode import unidecode

# 0) ENV
load_dotenv()

TELNYX_API_KEY        = os.getenv("TELNYX_API_KEY")
TELNYX_ASSISTANT_ID   = os.getenv("TELNYX_ASSISTANT_ID")
TELNYX_SIGNING_SECRET = os.getenv("TELNYX_SIGNING_SECRET", "")
HUMAN_FORWARD_NUMBER  = os.getenv("HUMAN_FORWARD_NUMBER")     # your Telnyx/E.164 staff number

# Apps Script webhook (already deployed by you; "anyone" or signed as you)
APPSCRIPT_WEBHOOK_URL = os.getenv("APPSCRIPT_WEBHOOK_URL")    # e.g. https://script.google.com/macros/s/AKfycb.../exec

# Optional tag to write into the sheet
SHEET_TAB             = os.getenv("GOOGLE_SHEETS_TAB_NAME", "Bookings")

# Business display
BUSINESS_NAME         = os.getenv("BUSINESS_NAME", "PlayStayTion Pet Resort and Training")
BUSINESS_CITY         = os.getenv("BUSINESS_CITY", "Sadler, TX")

# KB docx path
KB_DOCX_PATH          = os.getenv("KB_DOCX_PATH")
KB_MAX_CHARS          = int(os.getenv("KB_MAX_CHARS", "18000"))

print("=== ENV CHECK ===")
print("TELNYX_API_KEY set?        ", bool(TELNYX_API_KEY))
print("TELNYX_ASSISTANT_ID set?   ", bool(TELNYX_ASSISTANT_ID))
print("TELNYX_SIGNING_SECRET set? ", bool(TELNYX_SIGNING_SECRET))
print("HUMAN_FORWARD_NUMBER set?  ", bool(HUMAN_FORWARD_NUMBER))
print("APPSCRIPT_WEBHOOK_URL set? ", bool(APPSCRIPT_WEBHOOK_URL))
print("SHEET_TAB                  ", SHEET_TAB)
print("KB_DOCX_PATH               ", KB_DOCX_PATH)
print("======================================")

assert TELNYX_API_KEY, "Missing TELNYX_API_KEY"
assert TELNYX_ASSISTANT_ID, "Missing TELNYX_ASSISTANT_ID"
assert HUMAN_FORWARD_NUMBER, "Missing HUMAN_FORWARD_NUMBER"
assert APPSCRIPT_WEBHOOK_URL, "Missing APPSCRIPT_WEBHOOK_URL"

# 1) Load KB from Word (.docx)
def load_docx_as_text(path:str) -> str:
    doc = Document(path)
    chunks = []
    # paragraphs
    for p in doc.paragraphs:
        t = p.text.strip()
        if t:
            chunks.append(t)
    # simple tables (tab-separated)
    for tbl in doc.tables:
        for row in tbl.rows:
            cells = [c.text.strip() for c in row.cells]
            if any(cells):
                chunks.append("\t".join(cells))
    raw = "\n".join(chunks)
    raw = unidecode(raw)  # strip emojis/smart quotes
    raw = re.sub(r"[ \t]+", " ", raw)
    raw = re.sub(r"\n{3,}", "\n\n", raw).strip()
    return raw

try:
    KB_TEXT = load_docx_as_text(KB_DOCX_PATH)
    print("KB loaded:", KB_DOCX_PATH, "chars:", len(KB_TEXT))
except Exception as e:
    KB_TEXT = "KB not loaded. Please set KB_DOCX_PATH to a .docx file."
    print("KB load failed:", e)

# 2) Telnyx helpers
TELNYX_API_BASE = "https://api.telnyx.com/v2"
app = Flask(__name__)

def telnyx_post(path, payload):
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.post(url, headers={
        "Authorization": f"Bearer {TELNYX_API_KEY}",
        "Content-Type":"application/json"
    }, json=payload, timeout=25)
    print(f"POST {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    r.raise_for_status()
    return r.json()

def telnyx_patch(path, payload):
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.patch(url, headers={
        "Authorization": f"Bearer {TELNYX_API_KEY}",
        "Content-Type":"application/json"
    }, json=payload, timeout=25)
    print(f"PATCH {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    r.raise_for_status()
    return r.json()

def telnyx_get(path):
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.get(url, headers={"Authorization": f"Bearer {TELNYX_API_KEY}"}, timeout=25)
    print(f"GET {path} => {r.status_code}")
    if r.text: print(r.text[:700])
    return r

def verify_signature(raw, header):
    if not TELNYX_SIGNING_SECRET:
        return True
    digest = hmac.new(TELNYX_SIGNING_SECRET.encode(), raw, hashlib.sha256).hexdigest()
    return hmac.compare_digest(digest, header or "")

# 3) System prompt (same behavior, now with KB injection)

def build_prompt():
    kb_snippet = KB_TEXT.strip()
    if len(kb_snippet) > KB_MAX_CHARS:
        kb_snippet = kb_snippet[:KB_MAX_CHARS] + "\n[KB truncated for prompt size]\n"

    return f"""
You are a warm, patient voice receptionist for {BUSINESS_NAME} in {BUSINESS_CITY}.

TOP PRIORITIES
- Be kind, calm, and easy to follow. Use short sentences. Ask ONE question at a time, then pause.
- Speak as if the caller may not be tech-savvy. Offer simple explanations and to repeat things.
- Answer only from the Knowledge Base (KB) below. If a detail is missing, say you’re not certain and offer a human transfer.
- Collect booking details conversationally (one item per turn). Read back a short summary and confirm before saving.
- Never collect or store payment information. If the caller asks to pay, or starts giving card details, stop and TRANSFER to a human immediately.

PAYMENT & PRIVACY RULES (critical)
- Do NOT ask for or record credit cards, CVV, expiration dates, bank details, or any payment numbers.
- If the caller offers payment info: say you cannot take payment over the phone, and TRANSFER to a human.
- Only collect info needed to help (name, phone, service, date/time, pet details, notes). Avoid unnecessary personal data.

KNOWLEDGE BASE (authoritative; quote only from here, or say you’re not sure)
KB START
{kb_snippet}
KB END

INTAKE STYLE (strict)
- One detail per turn. Never list multiple questions at once.
- If the caller gives several details at once, briefly confirm what you captured, then ask the next missing item.
- Use gentle transitions: “Thanks. Next, what’s your dog’s name?”
- If the caller sounds unsure, explain options slowly using plain language, then ask a simple question.

FIELDS TO COLLECT (ask in this order; one per turn)
1) name (first name is fine)
2) phone (assume caller ID if not provided; read it back to confirm)
3) service (boarding | grooming | daycare | training | other)
4) date (YYYY-MM-DD or natural language like “this Friday”)
5) time (HH:MM or “morning / afternoon / evening”)
6) pet_name
7) breed
8) weight
9) notes (free text for meds, temperament, grooming style, special requests)
10) optional start_date and end_date for multi-day boarding, if relevant

ACCESSIBILITY & CLARITY
- Slow the pace with shorter prompts: “Got it. What’s your first name?”
- Offer to repeat or rephrase: “I can repeat that more slowly—would you like me to?”
- Confirm critical items (dates, phone numbers, pet name) by reading them back.
- If asked to text information or a link, use SEND_SMS with a short message.

VALIDATION & REPAIR
- Dates: If vague (“this weekend”), ask: “Which day should I put down?”
- Phone: If missing, ask for a callback number and read it back.
- Service: If unsure, offer the closest two options briefly and ask which fits best.

AFTER-HOURS OR SILENCE
- If two attempts get no response, offer to text a link or TRANSFER.
- For after-hours: collect what you can, then SAVE_LEAD and (optionally) SEND_SMS with a confirmation.

READBACK BEFORE SAVING
- When you have at least name, phone, service, and a date/time (or date range), read a short summary and ask: “Is that correct?” If yes, output the marker.

OUTPUT MARKERS (exact one-line format, then stop speaking immediately)
1) SAVE_LEAD{{"name":"<first>","phone":"<caller or provided>","service":"<boarding|grooming|daycare|training|other>","date":"<YYYY-MM-DD or text>","time":"<HH:MM or text>","pet_name":"<text>","breed":"<text>","weight":"<text>","notes":"<free text>","send_sms":true|false,"start_date":"<optional>","end_date":"<optional>"}}

2) TRANSFER{{"reason":"<short reason>","priority":"normal"|"urgent"}}

3) SEND_SMS{{"message":"<short confirmation or link>","to":"<E.164 or empty to use caller>"}}

Legacy support (accepted by backend):
READY_TO_TRANSFER{{"name":"...","phone":"...","service":"...","date":"...","time":"...","pet_name":"...","breed":"...","weight":"...","notes":"..."}}

MARKER RULES
- Emit exactly one marker line and nothing after it.
- Include every key; use "" if unknown.
- After emitting a marker, stop speaking and wait.

TONE EXAMPLES
Caller: “I want to book boarding.”
You: “Happy to help with boarding. I’ll keep it simple and quick. What’s your first name?”
(Wait)
You: “Thanks, Ana. What’s your dog’s name?”
(Wait)
You: “Great—Buddy. About how much does Buddy weigh?”
(Continue one item at a time. If they ask to pay: “I can’t take payments by phone, but I can connect you to a person who can.” Then TRANSFER.)
""".strip()


# 4) Apps Script writer + verification helpers ---

APPSCRIPT_TOKEN = os.getenv("APPSCRIPT_TOKEN", "")

def write_row_apps_script(row: dict):
    """
    POST row to Apps Script (action=append),
    verify response has ok:true and rowIndex.
    """
    payload = {
        "action": "append",
        "sheet": SHEET_TAB,
        "row": row,
        "token": APPSCRIPT_TOKEN or None
    }
    r = requests.post(APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script =>", r.status_code, r.text[:400])
    r.raise_for_status()
    resp = r.json()
    if not resp.get("ok"):
        raise RuntimeError(f"Apps Script append failed: {resp}")
    if not resp.get("rowIndex"):
        raise RuntimeError(f"Apps Script did not return rowIndex: {resp}")
    return resp  # contains sheet, rowIndex, rowCount, timestamp, nonce

def read_last_rows_from_apps_script(n=5):
    payload = {
        "action": "readLast",
        "sheet": SHEET_TAB,
        "n": n,
        "token": APPSCRIPT_TOKEN or None
    }
    r = requests.post(APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script readLast =>", r.status_code, r.text[:400])
    r.raise_for_status()
    return r.json()

@app.get("/gs/health")
def gs_health():
    """
    Writes a test probe row with a unique nonce and returns the append response.
    Use this to verify that the webhook writes to the Sheet.
    """
    nonce = f"probe-{int(time.time())}"
    row = {
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "name": "HealthCheck",
        "phone": "",
        "service": "healthcheck",
        "date": "",
        "time": "",
        "pet_name": "",
        "breed": "",
        "weight": "",
        "notes": "gs_health probe",
        "call_control_id": "",
        "start_date": "",
        "end_date": "",
        "source": "healthcheck",
        "nonce": nonce
    }
    try:
        resp = write_row_apps_script(row)
        return jsonify({"ok": True, "append_response": resp, "nonce": nonce})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/gs/check_last")
def gs_check_last():
    """
    Optional: read back the last N rows and (if 'nonce' is provided as a query param)
    confirm the probe exists.
    """
    n = int(request.args.get("n", "5"))
    nonce = request.args.get("nonce", "")
    try:
        data = read_last_rows_from_apps_script(n=n)
        found = False
        # rows is a list of arrays in the sheet's column order
        # we put 'Nonce' as the 15th column in the Apps Script header;
        # index 14 (0-based) should contain the nonce if present.
        for row in data.get("rows", []):
            if len(row) >= 15 and nonce and str(row[14]).strip() == nonce:
                found = True
                break
        return jsonify({"ok": True, "found_nonce": found, "nonce": nonce, "raw": data})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

# 5) Flask routes
@app.get("/health")
def health():
    return jsonify({"ok": True})

@app.get("/kb")
def kb_preview():
    return jsonify({"ok": True, "length": len(KB_TEXT), "snippet_first_800": KB_TEXT[:800]})

@app.post("/reload_kb")
def reload_kb():
    global KB_TEXT
    path = request.args.get("path") or KB_DOCX_PATH
    try:
        KB_TEXT = load_docx_as_text(path)
        return jsonify({"ok": True, "path": path, "length": len(KB_TEXT)})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/check-assistant")
def check_assistant():
    r = telnyx_get(f"/ai/assistants/{TELNYX_ASSISTANT_ID}")
    return (r.text, r.status_code, {"Content-Type":"application/json"})

def telnyx_api(method, path, payload=None):
    url = f"{TELNYX_API_BASE}{path}"
    headers = {"Authorization": f"Bearer {TELNYX_API_KEY}", "Content-Type":"application/json"}
    r = requests.request(method, url, headers=headers, json=payload, timeout=20)
    print(f"{method} {path} => {r.status_code}")
    if r.text:
        print(r.text[:1000])
    return r


# In section 5, update the /update_prompt route:

@app.get("/update_prompt")
def update_prompt():
    """Updates the Telnyx AI Assistant's instructions with the latest prompt and KB."""
    # Compose the core instructions and append the KB
    instructions = build_prompt()

    kb_suffix = ""
    if KB_TEXT:
        kb_content = KB_TEXT[:120000]
        kb_suffix = (
            "\n\n"
            "Knowledge Base (verbatim, highest authority):\n"
            "----\n"
            f"{kb_content}"
            "\n----\n"
            "Use these policies to answer. If a fact conflicts with prior memories, prefer this KB.\n"
        )
    instructions += kb_suffix

    # Try POST, then PUT (Removing PATCH since it consistently returns 404)
    paths_tried = []
    # Use POST as the primary method, then fallback to PUT if needed.
    for method in ("POST", "PUT"): 
        try:
            r = telnyx_api(method, f"/ai/assistants/{CONFIG.TELNYX_ASSISTANT_ID}", {"instructions": instructions})
            paths_tried.append((method, r.status_code))
            if r.status_code in (200, 201):
                return jsonify({"ok": True, "method": method, "status": r.status_code, "kb_len": len(KB_TEXT)})
        except requests.HTTPError as e:
            paths_tried.append((method, e.response.status_code))
            if e.response.status_code in (401, 403):
                 return jsonify({"ok": False, "error": "Telnyx auth/scope error updating assistant.", "tried": paths_tried}), 500

    # If all failed
    return jsonify({
        "ok": False,
        "error": "Could not update assistant via API (POST/PUT failed).",
        "tried": paths_tried
    }), 500

@app.post("/telnyx/webhook")
def telnyx_webhook():
    if not verify_signature(request.get_data(), request.headers.get("Telnyx-Signature")):
        return jsonify({"error": "bad signature"}), 400

    evt = request.get_json(silent=True) or {}
    data    = evt.get("data", {}) or {}
    payload = data.get("payload", {}) or {}
    etype   = data.get("event_type")
    ccid    = payload.get("call_control_id")
    caller  = payload.get("from")

    print("Event:", etype, "ccid:", ccid)

    # Answer + greeting
    if etype == "call.initiated" and ccid:
        try:
            telnyx_post(f"/calls/{ccid}/actions/answer", {})
            telnyx_post(f"/calls/{ccid}/actions/speak", {
                "language":"en-US","voice":"Telnyx.NaturalHD.astra",
                "payload": f"Hi, thanks for calling {BUSINESS_NAME}. One moment please."
            })
        except Exception as e:
            print("answer/speak err:", e)

    # Start the assistant
    if etype == "call.answered" and ccid:
        try:
            telnyx_post(f"/calls/{ccid}/actions/ai_assistant_start", {
                "assistant": {"id": TELNYX_ASSISTANT_ID}
            })
        except Exception as e:
            print("ai_assistant_start err:", e)

    # Scan conversation for markers
    if etype in ("call.conversation_insights.generated","call.conversation.updated","call.message.created"):
        texts = []

        for res in (payload.get("results") or []):
            if isinstance(res, dict) and isinstance(res.get("result"), str):
                texts.append(res["result"])

        for fld in ("transcript","text","message","content"):
            val = payload.get(fld)
            if isinstance(val, str):
                texts.append(val)

        for msg in (payload.get("messages") or []):
            if isinstance(msg, dict):
                for fld in ("text","content","message"):
                    if isinstance(msg.get(fld), str):
                        texts.append(msg[fld])

        if texts:
            print("Scanned last outputs:")
            for t in texts[-5:]:
                print("  •", t[:250].replace("\n"," "))

        re_save_lead = re.compile(r'\bSAVE_LEAD\{([^}]*)\}')
        re_transfer  = re.compile(r'\bTRANSFER\{([^}]*)\}')
        re_send_sms  = re.compile(r'\bSEND_SMS\{([^}]*)\}')
        re_ready_xfer= re.compile(r'\bREADY_TO_TRANSFER\{([^}]*)\}')

        found = {"SAVE_LEAD":None,"TRANSFER":None,"SEND_SMS":None,"READY_TO_TRANSFER":None}
        for t in texts:
            if not t: continue
            if not found["SAVE_LEAD"]:
                m = re_save_lead.search(t);  found["SAVE_LEAD"] = "{" + m.group(1) + "}" if m else None
            if not found["TRANSFER"]:
                m = re_transfer.search(t);   found["TRANSFER"]  = "{" + m.group(1) + "}" if m else None
            if not found["SEND_SMS"]:
                m = re_send_sms.search(t);   found["SEND_SMS"]  = "{" + m.group(1) + "}" if m else None
            if not found["READY_TO_TRANSFER"]:
                m = re_ready_xfer.search(t); found["READY_TO_TRANSFER"] = "{" + m.group(1) + "}" if m else None

        def parse_obj(s):
            try:
                return json.loads(s) if s else {}
            except Exception as e:
                print("Bad marker JSON:", e, s)
                return {}

        # legacy: save + transfer
        if found["READY_TO_TRANSFER"] and ccid:
            obj = parse_obj(found["READY_TO_TRANSFER"])
            if not obj.get("phone"):
                obj["phone"] = caller or ""
            row = {
                "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
                "name": obj.get("name",""),
                "phone": obj.get("phone",""),
                "service": obj.get("service",""),
                "date": obj.get("date",""),
                "time": obj.get("time",""),
                "pet_name": obj.get("pet_name",""),
                "breed": obj.get("breed",""),
                "weight": obj.get("weight",""),
                "notes": obj.get("notes",""),
                "call_control_id": ccid,
                "tab": SHEET_TAB,
                "source": "legacy_ready_to_transfer"
            }
            try:
                write_row_apps_script(row)
            except Exception as e:
                print("Apps Script write failed (legacy):", e)
            try:
                telnyx_post(f"/calls/{ccid}/actions/transfer", {"to": HUMAN_FORWARD_NUMBER})
                print("Transfer initiated (legacy) to", HUMAN_FORWARD_NUMBER)
            except Exception as e:
                print("Transfer err:", e)

        # SAVE_LEAD
        if found["SAVE_LEAD"]:
            obj = parse_obj(found["SAVE_LEAD"])
            row = {
                "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
                "name": obj.get("name",""),
                "phone": obj.get("phone") or caller or "",
                "service": obj.get("service",""),
                "date": obj.get("date",""),
                "time": obj.get("time",""),
                "pet_name": obj.get("pet_name",""),
                "breed": obj.get("breed",""),
                "weight": obj.get("weight",""),
                "notes": obj.get("notes",""),
                "call_control_id": ccid or "",
                "start_date": obj.get("start_date",""),
                "end_date": obj.get("end_date",""),
                "tab": SHEET_TAB,
                "source": "save_lead"
            }
            try:
                write_row_apps_script(row)
                print("Lead saved")
            except Exception as e:
                print("Apps Script write failed:", e)

            if obj.get("send_sms") and (obj.get("phone") or caller):
                try:
                    telnyx_post("/messages", {
                        "from": HUMAN_FORWARD_NUMBER,
                        "to": obj.get("phone") or caller,
                        "text": "Thanks! We saved your request and will follow up shortly."
                    })
                    print("Confirmation SMS sent")
                except Exception as e:
                    print("SMS err:", e)

        # TRANSFER
        if found["TRANSFER"] and ccid:
            obj = parse_obj(found["TRANSFER"])
            try:
                telnyx_post(f"/calls/{ccid}/actions/transfer", {"to": HUMAN_FORWARD_NUMBER})
                print("Transfer initiated:", obj.get("reason",""))
            except Exception as e:
                print("Transfer err:", e)

        # SEND_SMS
        if found["SEND_SMS"]:
            obj = parse_obj(found["SEND_SMS"])
            to = obj.get("to") or caller
            msg = (obj.get("message") or "").strip()
            if to and msg:
                try:
                    telnyx_post("/messages", {"from": HUMAN_FORWARD_NUMBER, "to": to, "text": msg})
                    print("SMS sent to", to)
                except Exception as e:
                    print("SMS err:", e)
            else:
                print("SEND_SMS missing to/message")

    return jsonify({"ok": True})

# 6) Launch Flask in background (Jupyter-safe)
def start_flask_background():
    th = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8080, debug=False, use_reloader=False))
    th.daemon = True
    th.start()
    time.sleep(1)
    print("Flask running on http://127.0.0.1:8080")
    return th

flask_thread = start_flask_background()

# 7) Cloudflared quick tunnel (no account)
def ensure_cloudflared():
    exe = "cloudflared.exe" if os.name == "nt" else "cloudflared"
    if pathlib.Path(exe).exists():
        return os.path.abspath(exe)
    print("Downloading cloudflared...")
    if os.name == "nt":
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
    elif sys.platform == "darwin":
        raise RuntimeError("On macOS, run: brew install cloudflared")
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    r = requests.get(url, timeout=60); r.raise_for_status()
    with open(exe, "wb") as f: f.write(r.content)
    if os.name != "nt":
        os.chmod(exe, os.stat(exe).st_mode | stat.S_IEXEC)
    return os.path.abspath(exe)

def start_cloudflared_quick_tunnel():
    exe = ensure_cloudflared()
    cmd = [exe, "tunnel", "--url", "http://localhost:8080", "--no-autoupdate"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    public_url = None
    for _ in range(240):
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.1); continue
        print("[cloudflared]", line.strip())
        m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break
    if not public_url:
        proc.terminate()
        raise RuntimeError("Could not detect cloudflared URL")
    return public_url, proc

public_url, cloudflared_proc = start_cloudflared_quick_tunnel()

print("\nBase:                 ", public_url)
print("Health:               ", public_url + "/health")
print("KB preview:           ", public_url + "/kb")
print("Reload KB (POST):     ", public_url + "/reload_kb")
print("Check assistant:      ", public_url + "/check-assistant")
print("Update prompt (GET):  ", public_url + "/update_prompt")
print("Webhook (POST):       ", public_url + "/telnyx/webhook")
print("\nIn Telnyx: Voice → Applications → Webhook URL =", public_url + "/telnyx/webhook")
print("Assign your DID to the SAME Voice Application, then place a real call.\n")

# Push latest prompt to Assistant now
# try:
#     resp = requests.get(public_url + "/update_prompt", timeout=15)
#     print("Update prompt:", resp.status_code, resp.text[:300])
# except Exception as e:
#     print("Update prompt call failed (non-fatal):", e)

# Push latest prompt to Assistant now (use local, avoids DNS races)
try:
    resp = requests.get("http://127.0.0.1:8080/update_prompt", timeout=15)
    print("Update prompt:", resp.status_code, resp.text[:300])
except Exception as e:
    print("Update prompt call failed (non-fatal):", e)


# Keep alive
try:
    while True:
        time.sleep(3600)
except KeyboardInterrupt:
    try:
        cloudflared_proc.terminate()
    except Exception:
        pass
    print("Stopped.")


In [ ]:
##GEMINI
import os
import json
import time
import hmac
import hashlib
import threading
import re
import subprocess
import sys
import stat
import pathlib
from dataclasses import dataclass
from typing import Optional, Any, Dict, List

import requests
from flask import Flask, request, jsonify
from dotenv import load_dotenv
from docx import Document
from unidecode import unidecode

# 0) Configuration and Environment
load_dotenv()

@dataclass
class Config:
    """Centralized configuration from environment variables."""
    # Telnyx
    TELNYX_API_KEY: str = os.getenv("TELNYX_API_KEY", "")
    TELNYX_ASSISTANT_ID: str = os.getenv("TELNYX_ASSISTANT_ID", "")
    TELNYX_SIGNING_SECRET: str = os.getenv("TELNYX_SIGNING_SECRET", "")
    HUMAN_FORWARD_NUMBER: str = os.getenv("HUMAN_FORWARD_NUMBER", "")
    TELNYX_API_BASE: str = "https://api.telnyx.com/v2"

    # Google Sheets / Apps Script
    APPSCRIPT_WEBHOOK_URL: str = os.getenv("APPSCRIPT_WEBHOOK_URL", "")
    APPSCRIPT_TOKEN: str = os.getenv("APPSCRIPT_TOKEN", "")
    SHEET_TAB: str = os.getenv("GOOGLE_SHEETS_TAB_NAME", "Bookings")

    # Business Display
    BUSINESS_NAME: str = os.getenv("BUSINESS_NAME", "PlayStayTion Pet Resort and Training")
    BUSINESS_CITY: str = os.getenv("BUSINESS_CITY", "Sadler, TX")
    RECEPTIONIST_NAME: str = "Taylor" # NEW: Explicitly define the receptionist name

    # Knowledge Base (KB)
    KB_DOCX_PATH: Optional[str] = os.getenv("KB_DOCX_PATH")
    KB_MAX_CHARS: int = int(os.getenv("KB_MAX_CHARS", "18000"))

    def __post_init__(self):
        """Perform required assertion checks after initialization."""
        if not self.TELNYX_API_KEY:
            raise ValueError("Missing TELNYX_API_KEY")
        if not self.TELNYX_ASSISTANT_ID:
            raise ValueError("Missing TELNYX_ASSISTANT_ID")
        if not self.HUMAN_FORWARD_NUMBER:
            raise ValueError("Missing HUMAN_FORWARD_NUMBER")
        if not self.APPSCRIPT_WEBHOOK_URL:
            raise ValueError("Missing APPSCRIPT_WEBHOOK_URL")

try:
    CONFIG = Config()
    print("=== ENV CHECK PASSED ===")
except ValueError as e:
    print(f"Configuration Error: {e}")
    sys.exit(1)


# 1) Load KB from Word (.docx)
def load_docx_as_text(path: str) -> str:
    """Loads text content from a .docx file, cleans it, and returns a single string."""
    if not path or not pathlib.Path(path).exists():
        print(f"KB not loaded. Path '{path}' not found.")
        return "KB not loaded. Please set KB_DOCX_PATH to a .docx file."

    try:
        doc = Document(path)
        chunks = []
        # paragraphs
        for p in doc.paragraphs:
            t = p.text.strip()
            if t:
                chunks.append(t)
        # simple tables (tab-separated)
        for tbl in doc.tables:
            for row in tbl.rows:
                cells = [c.text.strip() for c in row.cells]
                if any(cells):
                    chunks.append("\t".join(cells))

        raw = "\n".join(chunks)
        raw = unidecode(raw)
        raw = re.sub(r"[ \t]+", " ", raw)
        raw = re.sub(r"\n{3,}", "\n\n", raw).strip()
        return raw
    except Exception as e:
        print(f"KB load failed from {path}: {e}")
        return "KB not loaded due to error. Please check KB_DOCX_PATH."

KB_TEXT = load_docx_as_text(CONFIG.KB_DOCX_PATH)
print(f"KB loaded: {CONFIG.KB_DOCX_PATH} | Chars: {len(KB_TEXT)}")
print("======================================")


# 2) Telnyx helpers
app = Flask(__name__)

def telnyx_api(method: str, path: str, payload: Optional[Dict[str, Any]] = None) -> requests.Response:
    """Generic function for Telnyx API interaction (unified post, patch, get)."""
    url = f"{CONFIG.TELNYX_API_BASE}{path}"
    headers = {
        "Authorization": f"Bearer {CONFIG.TELNYX_API_KEY}",
        "Content-Type": "application/json"
    }

    try:
        r = requests.request(method, url, headers=headers, json=payload, timeout=25)
        print(f"TELNYX API: {method} {path} => {r.status_code}")
        if r.text:
            print(r.text[:500])

        # Raise for 4xx/5xx errors on mutating requests
        if method in ("POST", "PATCH", "PUT"):
            r.raise_for_status()

        return r
    except requests.exceptions.RequestException as e:
        print(f"ERROR: Telnyx API Request Failed ({method} {path}): {e}")
        raise

def verify_signature(raw_data: bytes, header: Optional[str]) -> bool:
    """Verifies the Telnyx webhook signature."""
    if not CONFIG.TELNYX_SIGNING_SECRET:
        return True
    digest = hmac.new(CONFIG.TELNYX_SIGNING_SECRET.encode(), raw_data, hashlib.sha256).hexdigest()
    return hmac.compare_digest(digest, header or "")


# 3) System prompt (RESTORING ORIGINAL STRUCTURE + NAMING)
def build_prompt() -> str:
    """
    Constructs the BASE system prompt with all behavioral rules and persona.
    The KB is dynamically appended in /update_prompt.
    """
    # Use Taylor in the prompt for consistent persona
    return f"""
You are {CONFIG.RECEPTIONIST_NAME}, a warm, patient voice receptionist for {CONFIG.BUSINESS_NAME} in {CONFIG.BUSINESS_CITY}.

TOP PRIORITIES
- Be kind, calm, and easy to follow. Use short sentences. Ask ONE question at a time, then pause.
- Speak as if the caller may not be tech-savvy. Offer simple explanations and to repeat things.
- Answer only from the **Knowledge Base (KB)** provided below this prompt. If a detail is missing, say you’re not certain and offer a human transfer.
- Collect booking details conversationally (one item per turn). Read back a short summary and confirm before saving.
- Never collect or store payment information. If the caller asks to pay, or starts giving card details, stop and TRANSFER to a human immediately.

PAYMENT & PRIVACY RULES (critical)
- Do NOT ask for or record credit cards, CVV, expiration dates, bank details, or any payment numbers.
- If the caller offers payment info: say you cannot take payment over the phone, and **TRANSFER** to a human.
- Only collect info needed to help (name, phone, service, date/time, pet details, notes). Avoid unnecessary personal data.

INTAKE STYLE (strict)
- One detail per turn. Never list multiple questions at once.
- If the caller gives several details at once, briefly confirm what you captured, then ask the next missing item.
- Use gentle transitions: “Thanks. Next, what’s your dog’s name?”
- If the caller sounds unsure, explain options slowly using plain language, then ask a simple question.

FIELDS TO COLLECT (ask in this order; one per turn)
1) name (first name is fine)
2) phone (assume caller ID if not provided; read it back to confirm)
3) service (boarding | grooming | daycare | training | other)
4) date (YYYY-MM-DD or natural language like “this Friday”)
5) time (HH:MM or “morning / afternoon / evening”)
6) pet_name
7) breed
8) weight
9) notes (free text for meds, temperament, grooming style, special requests)
10) optional start_date and end_date for multi-day boarding, if relevant

ACCESSIBILITY & CLARITY
- Slow the pace with shorter prompts: “Got it. What’s your first name?”
- Offer to repeat or rephrase: “I can repeat that more slowly—would you like me to?”
- Confirm critical items (dates, phone numbers, pet name) by reading them back.
- If asked to text information or a link, use SEND_SMS with a short message.

VALIDATION & REPAIR
- Dates: If vague (“this weekend”), ask: “Which day should I put down?”
- Phone: If missing, ask for a callback number and read it back.
- Service: If unsure, offer the closest two options briefly and ask which fits best.

AFTER-HOURS OR SILENCE
- If two attempts get no response, offer to text a link or TRANSFER.
- For after-hours: collect what you can, then SAVE_LEAD and (optionally) SEND_SMS with a confirmation.

READBACK BEFORE SAVING
- When you have at least name, phone, service, and a date/time (or date range), read a short summary and ask: “Is that correct?” If yes, output the marker.

OUTPUT MARKERS (exact one-line format, then stop speaking immediately)
1) SAVE_LEAD{{"name":"<first>","phone":"<caller or provided>","service":"<boarding|grooming|daycare|training|other>","date":"<YYYY-MM-DD or text>","time":"<HH:MM or text>","pet_name":"<text>","breed":"<text>","weight":"<text>","notes":"<free text>","send_sms":true|false,"start_date":"<optional>","end_date":"<optional>"}}

2) TRANSFER{{"reason":"<short reason>","priority":"normal"|"urgent"}}

3) SEND_SMS{{"message":"<short confirmation or link>","to":"<E.164 or empty to use caller>"}}

Legacy support (accepted by backend):
READY_TO_TRANSFER{{"name":"...","phone":"...","service":"...","date":"...","time":"...","pet_name":"...","breed":"...","weight":"...","notes":"..."}}

MARKER RULES
- Emit exactly one marker line and nothing after it.
- Include every key; use "" if unknown.
- After emitting a marker, stop speaking and wait.

TONE EXAMPLES
Caller: “I want to book boarding.”
You: “Happy to help with boarding. I’ll keep it simple and quick. What’s your first name?”
(Wait)
You: “Thanks, Ana. What’s your dog’s name?”
(Wait)
You: “Great—Buddy. About how much does Buddy weigh?”
(Continue one item at a time. If they ask to pay: “I can’t take payments by phone, but I can connect you to a person who can.” Then TRANSFER.)
""".strip()


# 4) Apps Script writer + verification helpers
def write_row_apps_script(row: dict):
    """POST row to Apps Script (action=append) and verify success."""
    payload = {
        "action": "append",
        "sheet": CONFIG.SHEET_TAB,
        "row": row,
        "token": CONFIG.APPSCRIPT_TOKEN or None
    }
    r = requests.post(CONFIG.APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script =>", r.status_code, r.text[:400])
    r.raise_for_status()
    resp = r.json()
    if not resp.get("ok") or not resp.get("rowIndex"):
        raise RuntimeError(f"Apps Script append failed or missing index: {resp}")
    return resp

def read_last_rows_from_apps_script(n=5):
    # Retained as-is
    payload = {
        "action": "readLast",
        "sheet": CONFIG.SHEET_TAB,
        "n": n,
        "token": CONFIG.APPSCRIPT_TOKEN or None
    }
    r = requests.post(CONFIG.APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script readLast =>", r.status_code, r.text[:400])
    r.raise_for_status()
    return r.json()

@app.get("/gs/health")
def gs_health():
    # Retained as-is, using CONFIG
    nonce = f"probe-{int(time.time())}"
    row = {
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "name": "HealthCheck", "phone": "", "service": "healthcheck",
        "date": "", "time": "", "pet_name": "", "breed": "",
        "weight": "", "notes": "gs_health probe",
        "call_control_id": "", "start_date": "", "end_date": "",
        "source": "healthcheck", "nonce": nonce
    }
    try:
        resp = write_row_apps_script(row)
        return jsonify({"ok": True, "append_response": resp, "nonce": nonce})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/gs/check_last")
def gs_check_last():
    # Retained as-is, using CONFIG
    n = int(request.args.get("n", "5"))
    nonce = request.args.get("nonce", "")
    try:
        data = read_last_rows_from_apps_script(n=n)
        found = False
        for row in data.get("rows", []):
            if len(row) >= 15 and nonce and str(row[14]).strip() == nonce:
                found = True
                break
        return jsonify({"ok": True, "found_nonce": found, "nonce": nonce, "raw": data})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500


# 5) Flask routes
@app.get("/health")
def health():
    return jsonify({"ok": True})

@app.get("/kb")
def kb_preview():
    return jsonify({"ok": True, "length": len(KB_TEXT), "snippet_first_800": KB_TEXT[:800]})

@app.post("/reload_kb")
def reload_kb():
    global KB_TEXT
    path = request.args.get("path") or CONFIG.KB_DOCX_PATH
    try:
        KB_TEXT = load_docx_as_text(path)
        return jsonify({"ok": True, "path": path, "length": len(KB_TEXT)})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/check-assistant")
def check_assistant():
    r = telnyx_api("GET", f"/ai/assistants/{CONFIG.TELNYX_ASSISTANT_ID}")
    return (r.text, r.status_code, {"Content-Type":"application/json"})

@app.get("/update_prompt")
def update_prompt():
    """Updates the Telnyx AI Assistant's instructions with the latest prompt and KB."""
    # Compose the core instructions
    instructions = build_prompt()

    # Append the KB
    kb_suffix = ""
    if KB_TEXT:
        # Use the maximum allowed for instructions (capped by a safe limit)
        kb_content = KB_TEXT[:120000]
        
        kb_suffix = (
            "\n\n"
            "Knowledge Base (verbatim, highest authority):\n"
            "----\n"
            f"{kb_content}"
            "\n----\n"
            "Use these policies to answer. If a fact conflicts with prior memories, prefer this KB.\n"
        )
    
    instructions += kb_suffix

    # Try PATCH, then POST, then PUT
    paths_tried = []
    for method in ("PATCH", "POST", "PUT"):
        try:
            r = telnyx_api(method, f"/ai/assistants/{CONFIG.TELNYX_ASSISTANT_ID}", {"instructions": instructions})
            paths_tried.append((method, r.status_code))
            if r.status_code in (200, 201):
                return jsonify({"ok": True, "method": method, "status": r.status_code, "kb_len": len(KB_TEXT)})
        except requests.HTTPError as e:
            paths_tried.append((method, e.response.status_code))
            if e.response.status_code in (401, 403):
                 return jsonify({"ok": False, "error": "Telnyx auth/scope error updating assistant.", "tried": paths_tried}), 500

    # If all failed
    return jsonify({
        "ok": False,
        "error": "Could not update assistant via API (PATCH/POST/PUT all failed).",
        "tried": paths_tried
    }), 500


@app.post("/telnyx/webhook")
def telnyx_webhook():
    """Handles Telnyx webhook events, processing call lifecycle and AI markers."""
    raw_data = request.get_data()
    if not verify_signature(raw_data, request.headers.get("Telnyx-Signature")):
        return jsonify({"error": "bad signature"}), 400

    evt = request.get_json(silent=True) or {}
    data = evt.get("data", {}) or {}
    payload = data.get("payload", {}) or {}
    etype = data.get("event_type")
    ccid = payload.get("call_control_id")
    caller = payload.get("from")

    print(f"WEBHOOK: Event: {etype}, CCID: {ccid}")

    # Answer + greeting (UPDATED FOR TAYLOR)
    if etype == "call.initiated" and ccid:
        try:
            telnyx_api("POST", f"/calls/{ccid}/actions/answer", {})
            
            # Personalized greeting
            new_greeting = f"Hi, thanks for calling {CONFIG.BUSINESS_NAME}. I'm {CONFIG.RECEPTIONIST_NAME}. How can I help you?"
            
            telnyx_api("POST", f"/calls/{ccid}/actions/speak", {
                "language": "en-US", 
                "voice": "Telnyx.NaturalHD.astra",
                "payload": new_greeting
            })
            
        except Exception as e:
            print(f"ERROR: answer/speak err: {e}")

    # Start the assistant
    elif etype == "call.answered" and ccid:
        try:
            telnyx_api("POST", f"/calls/{ccid}/actions/ai_assistant_start", {
                "assistant": {"id": CONFIG.TELNYX_ASSISTANT_ID}
            })
        except Exception as e:
            print(f"ERROR: ai_assistant_start err: {e}")

    # Scan conversation for markers
    elif etype in ("call.conversation_insights.generated", "call.conversation.updated", "call.message.created"):
        
        # --- Marker Detection Logic (Optimized for reliability and order) ---
        texts: List[str] = []
        for res in (payload.get("results") or []):
             if isinstance(res, dict) and isinstance(res.get("result"), str): texts.append(res["result"])
        for fld in ("transcript","text","message","content"):
            val = payload.get(fld)
            if isinstance(val, str): texts.append(val)
        for msg in (payload.get("messages") or []):
            if isinstance(msg, dict):
                for fld in ("text","content","message"):
                    if isinstance(msg.get(fld), str): texts.append(msg[fld])

        if not texts:
            return jsonify({"ok": True, "status": "no text to scan"})
            
        print("Scanned last outputs:", [t[:50].replace('\n', ' ') for t in texts[-3:]])

        def parse_marker(text: str, marker_name: str) -> Optional[Dict[str, Any]]:
            """Finds and parses a JSON marker from text."""
            pattern = re.compile(rf'\b{re.escape(marker_name)}\{{([^}}]*)\}}')
            match = pattern.search(text)
            if match:
                json_str = "{" + match.group(1) + "}"
                try:
                    # Robust parsing
                    return json.loads(json_str.replace("'", '"')) 
                except json.JSONDecodeError as e:
                    print(f"ERROR: Bad marker JSON for {marker_name}: {e}, string: {json_str}")
            return None

        found_markers = {}
        # Prioritize markers by searching from the END of the conversation
        for marker_name in ["TRANSFER", "SAVE_LEAD", "SEND_SMS", "READY_TO_TRANSFER"]:
            for t in reversed(texts):
                obj = parse_marker(t, marker_name)
                if obj:
                    found_markers[marker_name] = obj
                    break # Use the latest instance of this marker type

        # --- Marker Action Execution ---

        # 1. TRANSFER (Priority 1: Transfer immediately ends call processing)
        transfer_obj = found_markers.get("TRANSFER")
        if transfer_obj and ccid:
            try:
                telnyx_api("POST", f"/calls/{ccid}/actions/transfer", {"to": CONFIG.HUMAN_FORWARD_NUMBER})
                print("ACTION: Transfer initiated:", transfer_obj.get("reason",""))
                return jsonify({"ok": True, "action": "transfer"}) # Exit early after transfer
            except Exception as e:
                print(f"ERROR: Transfer err: {e}")
        
        # 2. SAVE_LEAD (Priority 2: Save and optionally send confirmation SMS)
        save_lead_obj = found_markers.get("SAVE_LEAD")
        if save_lead_obj:
            row = {
                "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
                "name": save_lead_obj.get("name",""),
                "phone": save_lead_obj.get("phone") or caller or "",
                "service": save_lead_obj.get("service",""),
                "date": save_lead_obj.get("date",""),
                "time": save_lead_obj.get("time",""),
                "pet_name": save_lead_obj.get("pet_name",""),
                "breed": save_lead_obj.get("breed",""),
                "weight": save_lead_obj.get("weight",""),
                "notes": save_lead_obj.get("notes",""),
                "call_control_id": ccid or "",
                "start_date": save_lead_obj.get("start_date",""),
                "end_date": save_lead_obj.get("end_date",""),
                "tab": CONFIG.SHEET_TAB,
                "source": "save_lead"
            }
            try:
                write_row_apps_script(row)
                print("ACTION: Lead saved (SAVE_LEAD)")
                
                if save_lead_obj.get("send_sms") and (row["phone"]):
                    telnyx_api("POST", "/messages", {
                        "from": CONFIG.HUMAN_FORWARD_NUMBER,
                        "to": row["phone"],
                        "text": "Thanks! We saved your request and will follow up shortly."
                    })
                    print("ACTION: Confirmation SMS sent")

            except Exception as e:
                print(f"ERROR: Lead save/SMS failed: {e}")
        
        # 3. READY_TO_TRANSFER (Legacy support: Save + Transfer)
        legacy_obj = found_markers.get("READY_TO_TRANSFER")
        if legacy_obj and ccid and not transfer_obj: # Don't re-transfer if a direct TRANSFER occurred
            row = {
                "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
                "name": legacy_obj.get("name",""),
                "phone": legacy_obj.get("phone") or caller or "",
                "service": legacy_obj.get("service",""),
                "date": legacy_obj.get("date",""),
                "time": legacy_obj.get("time",""),
                "pet_name": legacy_obj.get("pet_name",""),
                "breed": legacy_obj.get("breed",""),
                "weight": legacy_obj.get("weight",""),
                "notes": legacy_obj.get("notes",""),
                "call_control_id": ccid,
                "tab": CONFIG.SHEET_TAB,
                "source": "legacy_ready_to_transfer",
                "start_date": "", "end_date": ""
            }
            try:
                write_row_apps_script(row)
                print("ACTION: Lead saved (legacy READY_TO_TRANSFER)")
                telnyx_api("POST", f"/calls/{ccid}/actions/transfer", {"to": CONFIG.HUMAN_FORWARD_NUMBER})
                print("ACTION: Transfer initiated (legacy) to", CONFIG.HUMAN_FORWARD_NUMBER)
            except Exception as e:
                print(f"ERROR: Legacy save/transfer failed: {e}")

        # 4. SEND_SMS (Priority 3: Send a specific, non-confirmation SMS)
        sms_obj = found_markers.get("SEND_SMS")
        if sms_obj:
            to = sms_obj.get("to") or caller
            msg = (sms_obj.get("message") or "").strip()
            if to and msg:
                try:
                    telnyx_api("POST", "/messages", {"from": CONFIG.HUMAN_FORWARD_NUMBER, "to": to, "text": msg})
                    print(f"ACTION: SMS sent to {to}")
                except Exception as e:
                    print(f"ERROR: SMS send err: {e}")
            else:
                print("WARN: SEND_SMS marker missing 'to' or 'message'")

    return jsonify({"ok": True})

# 6) Launch Flask in background (Jupyter-safe)
def start_flask_background() -> threading.Thread:
    """Launches the Flask app in a background thread."""
    th = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8080, debug=False, use_reloader=False))
    th.daemon = True
    th.start()
    time.sleep(1) 
    print("\n[INFO] Flask running on http://127.0.0.1:8080")
    return th

# 7) Cloudflared quick tunnel
def ensure_cloudflared() -> str:
    # Retained as-is for tunneling functionality
    exe = "cloudflared.exe" if os.name == "nt" else "cloudflared"
    if pathlib.Path(exe).exists():
        return os.path.abspath(exe)
    print("Downloading cloudflared...")
    if os.name == "nt":
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
    elif sys.platform == "darwin":
        raise RuntimeError("On macOS, run: brew install cloudflared")
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    r = requests.get(url, timeout=60); r.raise_for_status()
    with open(exe, "wb") as f: f.write(r.content)
    if os.name != "nt":
        os.chmod(exe, os.stat(exe).st_mode | stat.S_IEXEC)
    return os.path.abspath(exe)

def start_cloudflared_quick_tunnel():
    # Retained as-is for tunneling functionality
    exe = ensure_cloudflared()
    cmd = [exe, "tunnel", "--url", "http://localhost:8080", "--no-autoupdate"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    public_url = None
    for _ in range(240):
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.1); continue
        print("[cloudflared]", line.strip())
        m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break
    if not public_url:
        proc.terminate()
        raise RuntimeError("Could not detect cloudflared URL")
    return public_url, proc

# Main execution
flask_thread = start_flask_background()

try:
    public_url, cloudflared_proc = start_cloudflared_quick_tunnel()

    print("\n--- DEPLOYMENT DETAILS ---")
    print(f"Receptionist: {CONFIG.RECEPTIONIST_NAME}")
    print("Base:                 ", public_url)
    print("Update prompt (GET):  ", public_url + "/update_prompt")
    print("Webhook (POST):       ", public_url + "/telnyx/webhook")
    print("\nACTION: Set Telnyx Voice App Webhook URL =", public_url + "/telnyx/webhook")
    print("ACTION: Call /update_prompt to sync AI instructions with new persona and KB.")
    print("--------------------------\n")
    
    # Push latest prompt to Assistant now (using local endpoint for robustness)
    resp = requests.get("http://127.0.0.1:8080/update_prompt", timeout=15)
    print(f"Update prompt result: {resp.status_code}. Response snippet: {resp.text[:300]}")

    # Keep alive
    while True:
        time.sleep(3600)
        
except KeyboardInterrupt:
    print("Stopping application...")
    if 'cloudflared_proc' in locals() and cloudflared_proc:
        try:
            cloudflared_proc.terminate()
        except Exception:
            pass
    print("Stopped.")
except Exception as e:
    print(f"\n[CRITICAL ERROR] Application failed to start or run: {e}")
    if 'cloudflared_proc' in locals() and cloudflared_proc:
        try:
            cloudflared_proc.terminate()
        except Exception:
            pass
    sys.exit(1)

=== ENV CHECK PASSED ===
KB loaded: C:\Users\chait\Documents\AI\Voice_Agent\PlayStation_Pet_Resort_Policy_Doc.docx | Chars: 22985
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://10.125.222.180:8080
Press CTRL+C to quit



[INFO] Flask running on http://127.0.0.1:8080
[cloudflared] 2025-10-22T16:14:00Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflared] 2025-10-22T16:14:00Z INF Requesting new quick Tunnel on trycloudflare.com...
[cloudflared] 2025-10-22T16:14:04Z INF +--------------------------------------------------------------------------------------------+
[cloudflared] 2025-10-22T16:14:04Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time

127.0.0.1 - - [22/Oct/2025 11:14:06] "GET /update_prompt HTTP/1.1" 200 -


TELNYX API: POST /ai/assistants/assistant-824811af-b3b2-4037-9bf9-a9024f11bc50 => 200
{"id":"assistant-824811af-b3b2-4037-9bf9-a9024f11bc50","name":"Voice_Agent","description":"","model":"Qwen/Qwen3-235B-A22B","instructions":"You are Taylor, a warm, patient voice receptionist for PlayStayTion Pet Resort and Training in Sadler, TX.\n\nTOP PRIORITIES\n- Be kind, calm, and easy to follow. Use short sentences. Ask ONE question at a time, then pause.\n- Speak as if the caller may not be tech-savvy. Offer simple explanations and to repeat things.\n- Answer only from the **Knowledge Bas
Update prompt result: 200. Response snippet: {"kb_len":22985,"method":"POST","ok":true,"status":200}



## FINAL CODE

In [ ]:
#!/usr/bin/env python3
"""
PlayStayTion Voice Agent (Telnyx AI Assistant)
- Injects Knowledge Base + intake instructions dynamically via ai_assistant_start
- Polls Telnyx conversation messages mid-call to catch markers reliably
- Robust marker parsing -> Google Sheets (Apps Script) write
- Cloudflared quick tunnel
- Behavior hammer to reduce repetition + enforce markers
- Readback "nudge" if user doesn't confirm
- Insights fallback -> partial row when model forgets marker
"""

import os, json, time, hmac, hashlib, re, subprocess, sys, stat, pathlib, requests, traceback
import threading
from threading import Event
from flask import Flask, request, jsonify
from dotenv import load_dotenv
from docx import Document
from unidecode import unidecode

# ========== 0) ENV ==========
load_dotenv()

TELNYX_API_KEY        = os.getenv("TELNYX_API_KEY")
TELNYX_ASSISTANT_ID   = os.getenv("TELNYX_ASSISTANT_ID")
TELNYX_SIGNING_SECRET = os.getenv("TELNYX_SIGNING_SECRET", "")
HUMAN_FORWARD_NUMBER  = os.getenv("HUMAN_FORWARD_NUMBER")     # Telnyx/E.164 staff number

APPSCRIPT_WEBHOOK_URL = os.getenv("APPSCRIPT_WEBHOOK_URL")    # https://script.google.com/macros/s/AKfycb.../exec
APPSCRIPT_TOKEN       = os.getenv("APPSCRIPT_TOKEN", "")

SHEET_TAB             = os.getenv("GOOGLE_SHEETS_TAB_NAME", "Bookings")

BUSINESS_NAME         = os.getenv("BUSINESS_NAME", "PlayStayTion Pet Resort and Training")
BUSINESS_CITY         = os.getenv("BUSINESS_CITY", "Sadler, TX")

KB_DOCX_PATH          = os.getenv("KB_DOCX_PATH")
KB_MAX_CHARS          = int(os.getenv("KB_MAX_CHARS", "18000"))   # max KB injected per call

print("=== ENV CHECK ===")
print("TELNYX_API_KEY set?         ", bool(TELNYX_API_KEY))
print("TELNYX_ASSISTANT_ID set?    ", bool(TELNYX_ASSISTANT_ID))
print("TELNYX_SIGNING_SECRET set?  ", bool(TELNYX_SIGNING_SECRET))
print("HUMAN_FORWARD_NUMBER set?   ", bool(HUMAN_FORWARD_NUMBER))
print("APPSCRIPT_WEBHOOK_URL set?  ", bool(APPSCRIPT_WEBHOOK_URL))
print("APPSCRIPT_TOKEN set?        ", bool(APPSCRIPT_TOKEN))
print("SHEET_TAB                   ", SHEET_TAB)
print("KB_DOCX_PATH                ", KB_DOCX_PATH)
print("KB_MAX_CHARS                ", KB_MAX_CHARS)
print("======================================")

assert TELNYX_API_KEY, "Missing TELNYX_API_KEY"
assert TELNYX_ASSISTANT_ID, "Missing TELNYX_ASSISTANT_ID"
assert HUMAN_FORWARD_NUMBER, "Missing HUMAN_FORWARD_NUMBER"
assert APPSCRIPT_WEBHOOK_URL, "Missing APPSCRIPT_WEBHOOK_URL"

# ========== 1) Load KB (.docx) ==========
def load_docx_as_text(path: str) -> str:
    if not path:
        return ""
    doc = Document(path)
    chunks = []
    for p in doc.paragraphs:
        t = p.text.strip()
        if t:
            chunks.append(t)
    for tbl in doc.tables:
        for row in tbl.rows:
            cells = [c.text.strip() for c in row.cells]
            if any(cells):
                chunks.append("\t".join(cells))
    raw = "\n".join(chunks)
    raw = unidecode(raw)  # remove unusual unicode
    raw = re.sub(r"[ \t]+", " ", raw)
    raw = re.sub(r"\n{3,}", "\n\n", raw).strip()
    return raw

try:
    KB_TEXT = load_docx_as_text(KB_DOCX_PATH)
    print("KB loaded:", KB_DOCX_PATH, "chars:", len(KB_TEXT))
except Exception as e:
    KB_TEXT = ""
    print("KB load failed:", e)

# ========== 2) Telnyx helpers ==========
TELNYX_API_BASE = "https://api.telnyx.com/v2"
app = Flask(__name__)

def telnyx_post(path, payload):
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.post(url, headers={
        "Authorization": f"Bearer {TELNYX_API_KEY}",
        "Content-Type":"application/json"
    }, json=payload, timeout=25)
    print(f"POST {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    r.raise_for_status()
    return r.json()

def telnyx_get(path):
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.get(url, headers={"Authorization": f"Bearer {TELNYX_API_KEY}"}, timeout=25)
    print(f"GET {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    return r

def telnyx_get_json(path):
    """GET helper that returns JSON or {} and logs HTTP code."""
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.get(url, headers={"Authorization": f"Bearer {TELNYX_API_KEY}"}, timeout=25)
    print(f"GET {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    try:
        r.raise_for_status()
        return r.json() if r.text else {}
    except Exception as e:
        print("GET error:", e)
        return {}

def verify_signature(raw, header):
    if not TELNYX_SIGNING_SECRET:
        return True
    digest = hmac.new(TELNYX_SIGNING_SECRET.encode(), raw, hashlib.sha256).hexdigest()
    return hmac.compare_digest(digest, header or "")

# ===== Conversation polling & state =====
_active_polls = {}           # { conversation_id: {"stop": Event, "last_seen_ids": set(), "ccid": str, "caller": str} }
_saved_conversations = set() # conversations where we already saved a row (marker or fallback)

# ========== 3) Prompts ==========
def build_prompt():
    kb_snippet = (KB_TEXT or "").strip()
    if len(kb_snippet) > KB_MAX_CHARS:
        kb_snippet = kb_snippet[:KB_MAX_CHARS] + "\n[KB truncated]\n"

    base = f"""
You are a warm, patient voice receptionist for {BUSINESS_NAME} in {BUSINESS_CITY}.

TOP PRIORITIES
- Be kind, calm, and easy to follow. Use short sentences. Ask ONE question at a time, then pause.
- Speak as if the caller may not be tech-savvy. Offer simple explanations and to repeat things.
- Answer only from the Knowledge Base (KB) below. If a detail is missing, say you’re not certain and offer a human transfer.
- Collect booking details conversationally (one item per turn). Read back a short summary and confirm before saving.
- Never collect or store payment information. If the caller asks to pay, or starts giving card details, stop and TRANSFER to a human immediately.

PAYMENT & PRIVACY RULES (critical)
- Do NOT ask for or record credit cards, CVV, expiration dates, bank details, or any payment numbers.
- If the caller offers payment info: say you cannot take payment over the phone, and TRANSFER to a human.
- Only collect info needed to help (name, phone, service, date/time, pet details, notes). Avoid unnecessary personal data.

IMPORTANT INSTRUCTION (HIGH PRIORITY)
- AFTER EMITTING ANY MARKER (SAVE_LEAD / TRANSFER / SEND_SMS / READY_TO_TRANSFER), IMMEDIATELY STOP SPEAKING. EMIT THE EXACT ONE-LINE MARKER AND THEN HALT — NO MORE WORDS OR AUDIO.

KNOWLEDGE BASE (authoritative; quote only from here, or say you’re not sure)
KB START
{kb_snippet}
KB END

INTAKE STYLE (strict)
- One detail per turn. Never list multiple questions at once.
- If the caller gives several details at once, briefly confirm what you captured, then ask the next missing item.
- Use gentle transitions: “Thanks. Next, what’s your dog’s name?”
- If the caller sounds unsure, explain options slowly in plain language, then ask a simple question.

FIELDS TO COLLECT (ask in this order; one per turn)
1) name (first name is fine)
2) phone (assume caller ID if not provided; read it back to confirm)
3) service (boarding | grooming | daycare | training | other)
4) date (YYYY-MM-DD or natural language like “this Friday”)
5) time (HH:MM or “morning / afternoon / evening”)
6) pet_name
7) breed
8) weight
9) notes (free text for meds, temperament, grooming style, special requests)
10) optional start_date and end_date for multi-day boarding, if relevant

READBACK BEFORE SAVING
- When you have at least name, phone, service, and a date/time (or date range), read a short summary and ask: “Is that correct?” If yes, output the marker.

OUTPUT MARKERS (exact one-line format, then stop speaking immediately)
1) SAVE_LEAD{{"name":"<first>","phone":"<caller or provided>","service":"<boarding|grooming|daycare|training|other>","date":"<YYYY-MM-DD or text>","time":"<HH:MM or text>","pet_name":"<text>","breed":"<text>","weight":"<text>","notes":"<free text>","send_sms":true|false,"start_date":"<optional>","end_date":"<optional>"}}
2) TRANSFER{{"reason":"<short reason>","priority":"normal"|"urgent"}}
3) SEND_SMS{{"message":"<short confirmation or link>","to":"<E.164 or empty to use caller>"}}
4) READY_TO_TRANSFER{{"name":"...","phone":"...","service":"...","date":"...","time":"...","pet_name":"...","breed":"...","weight":"...","notes":"..."}}

MARKER RULES
- Emit exactly one marker line and nothing after it.
- Include every key; use "" if unknown.
- After emitting a marker, stop speaking and wait.
""".strip()
    return base

def build_live_instructions():
    """Compose the exact instructions + trimmed KB we send on ai_assistant_start for THIS call."""
    core = build_prompt()
    kb_content = (KB_TEXT or "")[:KB_MAX_CHARS]
    core += (
        "\n\nKnowledge Base (verbatim, highest authority):\n----\n"
        f"{kb_content}\n"
        "----\n"
        "AFTER EMITTING ANY MARKER (SAVE_LEAD / TRANSFER / SEND_SMS / READY_TO_TRANSFER), "
        "PRINT THE EXACT ONE-LINE MARKER AND STOP. PRODUCE NO FURTHER WORDS OR AUDIO.\n"
        "\nBEHAVIOR OVERRIDES (FINAL, DO THIS EXACTLY)\n"
        "- Do not say “I didn’t catch that” more than once in the entire call. If unclear, rephrase once, then move to a simpler question or offer transfer.\n"
        "- Start intake proactively. First line after greeting: “Happy to help. What’s your first name?” Then proceed through FIELDS TO COLLECT, one item per turn.\n"
        "- When you read back the short summary and the caller says “Yes,” immediately output ONLY the one-line SAVE_LEAD{...} marker and then STOP with no additional words or audio.\n"
        "- If the caller attempts to pay or gives card numbers, output ONLY: TRANSFER{\"reason\":\"payment over phone\",\"priority\":\"urgent\"} and STOP.\n"
    )
    return core

# ========== 4) Apps Script helpers ==========
def write_row_apps_script(row: dict):
    """
    POST row to Apps Script (action=append). Accept ok:true even if rowIndex is missing.
    Optionally verify by readLast when rowIndex is absent.
    """
    payload = {
        "action": "append",
        "sheet": SHEET_TAB,
        "row": row,
        "token": APPSCRIPT_TOKEN or None
    }
    r = requests.post(APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script =>", r.status_code, r.text[:400])
    r.raise_for_status()
    resp = r.json()

    if not resp.get("ok"):
        raise RuntimeError(f"Apps Script append failed: {resp}")

    # Accept success without rowIndex
    if resp.get("rowIndex"):
        return resp

    # Optional verification
    try:
        confirm = read_last_rows_from_apps_script(n=5)
        stamp = row.get("timestamp_utc", "")
        name  = row.get("name", "")
        svc   = row.get("service", "")
        found = False
        for tail in confirm.get("rows", []):
            s = [str(c) for c in tail]
            if stamp and any(stamp in c for c in s):
                found = True; break
            if name and svc and any(name in c for c in s) and any(svc in c for c in s):
                found = True; break
        if found:
            print("Apps Script append verified via readLast (no rowIndex provided).")
            return {"ok": True, "verified_by_readLast": True}
    except Exception as e:
        print("Apps Script verification skip/error:", e)

    # Treat ok:true as success to avoid dropping leads
    return {"ok": True, "rowIndex": None, "note": "rowIndex not provided by Apps Script"}

def read_last_rows_from_apps_script(n=5):
    payload = {"action": "readLast", "sheet": SHEET_TAB, "n": n, "token": APPSCRIPT_TOKEN or None}
    r = requests.post(APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script readLast =>", r.status_code, r.text[:400])
    r.raise_for_status()
    return r.json()

# ========== 5) Marker scanning, nudge & poller ==========
READBACK_PATTERNS = ("Is that correct", "Does that look right", "Should I save that", "Is this okay")

def _scan_texts_for_markers(texts):
    re_save_lead = re.compile(r'SAVE_LEAD\{(.+?)\}', re.DOTALL)
    re_transfer  = re.compile(r'TRANSFER\{(.+?)\}', re.DOTALL)
    re_send_sms  = re.compile(r'SEND_SMS\{(.+?)\}', re.DOTALL)
    re_ready_xfer= re.compile(r'READY_TO_TRANSFER\{(.+?)\}', re.DOTALL)

    joined = "\n\n".join([t for t in texts if t])
    found = {"SAVE_LEAD":None,"TRANSFER":None,"SEND_SMS":None,"READY_TO_TRANSFER":None}
    if joined:
        m = re_save_lead.search(joined);  found["SAVE_LEAD"] = "{" + m.group(1).strip() + "}" if m else None
        m = re_transfer.search(joined);   found["TRANSFER"]  = "{" + m.group(1).strip() + "}" if m else None
        m = re_send_sms.search(joined);   found["SEND_SMS"]  = "{" + m.group(1).strip() + "}" if m else None
        m = re_ready_xfer.search(joined); found["READY_TO_TRANSFER"] = "{" + m.group(1).strip() + "}" if m else None
    return found

def _parse_marker_json(s):
    if not s: return {}
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(s.replace("'", '"'))
        except Exception as e:
            print("Marker JSON parse failed:", e, "| raw:", s[:400])
            return {}

def _nudge_if_readback_detected(texts, ccid):
    try:
        joined = " ".join(texts).lower()
        if any(p.lower() in joined for p in READBACK_PATTERNS):
            def _timer():
                time.sleep(15)
                try:
                    telnyx_post(f"/calls/{ccid}/actions/speak", {
                        "language":"en-US","voice":"Telnyx.NaturalHD.astra",
                        "payload":"Please say yes or no so I can save this for you."
                    })
                except Exception as e:
                    print("nudge speak err:", e)
            threading.Thread(target=_timer, daemon=True).start()
    except Exception as e:
        print("nudge check err:", e)

def _handle_found_markers(found, ccid, caller, conversation_id=None):
    global _saved_conversations
    # READY_TO_TRANSFER (legacy)
    if found.get("READY_TO_TRANSFER") and ccid:
        obj = _parse_marker_json(found["READY_TO_TRANSFER"])
        if not obj.get("phone"): obj["phone"] = caller or ""
        row = {
            "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "name": obj.get("name",""),
            "phone": obj.get("phone",""),
            "service": obj.get("service",""),
            "date": obj.get("date",""),
            "time": obj.get("time",""),
            "pet_name": obj.get("pet_name",""),
            "breed": obj.get("breed",""),
            "weight": obj.get("weight",""),
            "notes": obj.get("notes",""),
            "call_control_id": ccid,
            "start_date": obj.get("start_date",""),
            "end_date": obj.get("end_date",""),
            "tab": SHEET_TAB,
            "source": "legacy_ready_to_transfer"
        }
        try:
            write_row_apps_script(row)
            if conversation_id: _saved_conversations.add(conversation_id)
        except Exception as e:
            print("Apps Script write failed (legacy):", e)
        try:
            telnyx_post(f"/calls/{ccid}/actions/transfer", {"to": HUMAN_FORWARD_NUMBER})
            print("Transfer initiated (legacy) to", HUMAN_FORWARD_NUMBER)
        except Exception as e:
            print("Transfer err:", e)

    # SAVE_LEAD
    if found.get("SAVE_LEAD"):
        obj = _parse_marker_json(found["SAVE_LEAD"])
        row = {
            "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "name": obj.get("name",""),
            "phone": obj.get("phone") or caller or "",
            "service": obj.get("service",""),
            "date": obj.get("date",""),
            "time": obj.get("time",""),
            "pet_name": obj.get("pet_name",""),
            "breed": obj.get("breed",""),
            "weight": obj.get("weight",""),
            "notes": obj.get("notes",""),
            "call_control_id": ccid or "",
            "start_date": obj.get("start_date",""),
            "end_date": obj.get("end_date",""),
            "tab": SHEET_TAB,
            "source": "save_lead"
        }
        try:
            write_row_apps_script(row)
            print("Lead saved (poller/webhook)")
            if conversation_id: _saved_conversations.add(conversation_id)
        except Exception as e:
            print("Apps Script write failed:", e)

        if obj.get("send_sms") and (obj.get("phone") or caller):
            try:
                telnyx_post("/messages", {
                    "from": HUMAN_FORWARD_NUMBER,
                    "to": obj.get("phone") or caller,
                    "text": "Thanks! We saved your request and will follow up shortly."
                })
                print("Confirmation SMS sent")
            except Exception as e:
                print("SMS err:", e)

    # TRANSFER
    if found.get("TRANSFER") and ccid:
        obj = _parse_marker_json(found["TRANSFER"])
        try:
            telnyx_post(f"/calls/{ccid}/actions/transfer", {"to": HUMAN_FORWARD_NUMBER})
            print("Transfer initiated:", obj.get("reason",""))
        except Exception as e:
            print("Transfer err:", e)

    # SEND_SMS
    if found.get("SEND_SMS"):
        obj = _parse_marker_json(found["SEND_SMS"])
        to = obj.get("to") or caller
        msg = (obj.get("message") or "").strip()
        if to and msg:
            try:
                telnyx_post("/messages", {"from": HUMAN_FORWARD_NUMBER, "to": to, "text": msg})
                print("SMS sent to", to)
            except Exception as e:
                print("SMS err:", e)
        else:
            print("SEND_SMS missing to/message")

def _poll_conversation_messages(conversation_id, ccid, caller):
    """Fetch assistant messages periodically and look for markers."""
    print(f"[poller] start for conversation {conversation_id}")
    stop_ev = _active_polls[conversation_id]["stop"]
    last_seen = _active_polls[conversation_id]["last_seen_ids"]

    try:
        while not stop_ev.is_set():
            data = telnyx_get_json(f"/ai/conversations/{conversation_id}/messages")
            msgs = (data.get("data") or []) if isinstance(data, dict) else []
            texts = []
            new_ids = 0
            for m in msgs:
                mid = m.get("id")
                if mid and mid in last_seen:
                    continue
                if mid:
                    last_seen.add(mid)
                    new_ids += 1
                t = m.get("text") or m.get("content") or m.get("message") or ""
                if isinstance(t, str) and t.strip():
                    texts.append(t)

            if new_ids:
                print(f"[poller] {new_ids} new message(s) for {conversation_id}")
                if texts:
                    print("[poller] sample:", (texts[-1][:300].replace("\n"," ")))
                _nudge_if_readback_detected(texts, ccid)
                found = _scan_texts_for_markers(texts)
                if any(found.values()):
                    print("[poller] marker(s) detected:", [k for k,v in found.items() if v])
                    _handle_found_markers(found, ccid, caller, conversation_id)

            stop_ev.wait(1.8)
    finally:
        print(f"[poller] stop for conversation {conversation_id}")

# ========== 6) Insights fallback ==========
def _extract_fields_from_insight(text: str, caller: str):
    """Very light heuristic extraction for fallback rows."""
    service = ""
    for kw in ("boarding","grooming","daycare","training"):
        if kw in text.lower():
            service = kw; break

    # naive phone digits if mentioned, else caller
    phone_match = re.search(r'(?:\+?\d[\d\-\s]{7,}\d)', text)
    phone = re.sub(r'\D','', phone_match.group(0)) if phone_match else (caller or "")
    if phone and not phone.startswith("+") and len(phone) >= 10:
        phone = "+" + phone[-10:]

    # try pet name (capitalized word before "weigh"/"weight"/"dog"/"cat")
    pet_name = ""
    m = re.search(r'([A-Z][a-z]{1,20})[^.!?]{0,30}\b(?:dog|cat|pet|weigh|weight)', text)
    if m: pet_name = m.group(1)

    # try weight (first number + lb/lbs/pounds)
    weight = ""
    m = re.search(r'(\d{1,3})(?:\s?(?:lb|lbs|pounds)?)', text.lower())
    if m: weight = m.group(1)

    name = ""
    # cheap guess: first capitalized word at sentence start not matching pet name
    m = re.search(r'([A-Z][a-z]{1,20})\b', text)
    if m and m.group(1) != pet_name:
        name = m.group(1)

    return {
        "name": name,
        "phone": phone,
        "service": service or "other",
        "pet_name": pet_name,
        "weight": weight
    }

def _insights_fallback_append(conversation_id, ccid, caller, text):
    """Append a partial row if no marker was captured for this conversation."""
    if conversation_id in _saved_conversations:
        return  # already saved by marker
    fields = _extract_fields_from_insight(text or "", caller)
    row = {
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "name": fields.get("name",""),
        "phone": fields.get("phone") or caller or "",
        "service": fields.get("service","other"),
        "date": "",
        "time": "",
        "pet_name": fields.get("pet_name",""),
        "breed": "",
        "weight": fields.get("weight",""),
        "notes": f"[insights_fallback] {text[:180]}",
        "call_control_id": ccid or "",
        "start_date": "",
        "end_date": "",
        "tab": SHEET_TAB,
        "source": "insights_fallback",
        "nonce": f"insights-{int(time.time())}"
    }
    try:
        write_row_apps_script(row)
        _saved_conversations.add(conversation_id)
        print("Fallback row appended from insights.")
    except Exception as e:
        print("Apps Script write failed (insights fallback):", e)

# ========== 7) Flask routes ==========
@app.get("/health")
def health():
    return jsonify({"ok": True})

@app.get("/kb")
def kb_preview():
    return jsonify({"ok": True, "length": len(KB_TEXT), "snippet_first_800": (KB_TEXT or "")[:800]})

@app.post("/reload_kb")
def reload_kb():
    global KB_TEXT
    path = request.args.get("path") or KB_DOCX_PATH
    try:
        KB_TEXT = load_docx_as_text(path)
        return jsonify({"ok": True, "path": path, "length": len(KB_TEXT)})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/check-assistant")
def check_assistant():
    r = telnyx_get(f"/ai/assistants/{TELNYX_ASSISTANT_ID}")
    return (r.text, r.status_code, {"Content-Type":"application/json"})

# We don't patch the portal assistant; we inject live per call.
@app.get("/update_prompt")
def update_prompt():
    return jsonify({
        "ok": True,
        "note": "Skipping portal update. Instructions are injected dynamically via ai_assistant_start for each call.",
        "kb_len": len((KB_TEXT or "")[:KB_MAX_CHARS])
    })

@app.get("/gs/health")
def gs_health():
    nonce = f"probe-{int(time.time())}"
    row = {
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "name": "HealthCheck",
        "phone": "",
        "service": "healthcheck",
        "date": "",
        "time": "",
        "pet_name": "",
        "breed": "",
        "weight": "",
        "notes": "gs_health probe",
        "call_control_id": "",
        "start_date": "",
        "end_date": "",
        "source": "healthcheck",
        "nonce": nonce
    }
    try:
        resp = write_row_apps_script(row)
        return jsonify({"ok": True, "append_response": resp, "nonce": nonce})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/gs/check_last")
def gs_check_last():
    n = int(request.args.get("n", "5"))
    nonce = request.args.get("nonce", "")
    try:
        data = read_last_rows_from_apps_script(n=n)
        found = False
        for row in data.get("rows", []):
            if nonce and nonce in [str(c) for c in row]:
                found = True
                break
        return jsonify({"ok": True, "found_nonce": found, "nonce": nonce, "raw": data})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

# ========== 8) Telnyx webhook ==========
@app.post("/telnyx/webhook")
def telnyx_webhook():
    raw_body = request.get_data()
    sig = request.headers.get("Telnyx-Signature")
    if not verify_signature(raw_body, sig):
        return jsonify({"error": "bad signature"}), 400

    evt = request.get_json(silent=True) or {}
    data    = (evt.get("data") or {})
    payload = (data.get("payload") or {})
    etype   = data.get("event_type")
    ccid    = payload.get("call_control_id")
    caller  = payload.get("from")
    conv_id = (payload or {}).get("conversation_id")

    # Verbose log
    print("\n== Telnyx webhook event ==")
    try:
        print(json.dumps(evt, indent=2)[:4000])
    except Exception:
        print(str(evt)[:2000])
    print("Event:", etype, "ccid:", ccid, "from:", caller)

    # Greeting
    if etype == "call.initiated" and ccid:
        try:
            telnyx_post(f"/calls/{ccid}/actions/answer", {})
            telnyx_post(f"/calls/{ccid}/actions/speak", {
                "language":"en-US","voice":"Telnyx.NaturalHD.astra",
                "payload": f"Hi, thanks for calling {BUSINESS_NAME}. One moment please."
            })
        except Exception as e:
            print("answer/speak err:", e); traceback.print_exc()

    # Start assistant with live instructions
    if etype == "call.answered" and ccid:
        try:
            live_instructions = build_live_instructions()
            resp = telnyx_post(f"/calls/{ccid}/actions/ai_assistant_start", {
                "assistant": {
                    "id": TELNYX_ASSISTANT_ID,
                    "instructions": live_instructions
                }
            })
            print("ai_assistant_start response:", resp)
        except Exception as e:
            print("ai_assistant_start err:", e); traceback.print_exc()

    # When conversation is created, start poller
    if etype == "call.conversation.created":
        conv_id = (payload or {}).get("conversation_id")
        if conv_id and ccid and conv_id not in _active_polls:
            stop_ev = Event()
            _active_polls[conv_id] = {"stop": stop_ev, "last_seen_ids": set(), "ccid": ccid, "caller": payload.get("from")}
            threading.Thread(
                target=_poll_conversation_messages,
                args=(conv_id, ccid, payload.get("from")),
                daemon=True
            ).start()

    # Mid-call: parse any message-like webhooks (best-effort)
    if etype in (
        "call.conversation_insights.generated",
        "call.conversation.updated",
        "call.message.created",
        "ai_assistant.message.created",
        "ai_assistant.conversation.updated",
    ):
        texts = []
        for res in (payload.get("results") or []):
            if isinstance(res, dict) and isinstance(res.get("result"), str):
                texts.append(res["result"])
        for fld in ("transcript","text","message","content"):
            val = payload.get(fld)
            if isinstance(val, str):
                texts.append(val)
        for msg in (payload.get("messages") or []):
            if isinstance(msg, dict):
                for fld in ("text","content","message"):
                    if isinstance(msg.get(fld), str):
                        texts.append(msg[fld])

        if texts:
            print("Scanned last outputs (most recent 10):")
            for t in texts[-10:]:
                print("  •", (t[:250].replace("\n", " ")))

            _nudge_if_readback_detected(texts, ccid)
            found = _scan_texts_for_markers(texts)
            if any(found.values()):
                print("[webhook] marker(s) detected:", [k for k,v in found.items() if v])
                _handle_found_markers(found, ccid, caller, conv_id)

        # Insights fallback (when present)
        if etype == "call.conversation_insights.generated":
            try:
                insight_texts = [r.get("result","") for r in (payload.get("results") or []) if isinstance(r, dict)]
                insight_blob = " ".join(insight_texts).strip()
                if insight_blob and conv_id and conv_id not in _saved_conversations:
                    _insights_fallback_append(conv_id, ccid, caller, insight_blob)
            except Exception as e:
                print("insights fallback err:", e)

    # Stop poller on end/hangup
    if etype in ("call.conversation.ended", "call.hangup"):
        conv_id = (payload or {}).get("conversation_id")
        if conv_id and conv_id in _active_polls:
            _active_polls[conv_id]["stop"].set()
            try: del _active_polls[conv_id]
            except Exception: pass

    return jsonify({"ok": True})

# ========== 9) Launch Flask (Jupyter-safe) ==========
def start_flask_background():
    th = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8080, debug=False, use_reloader=False))
    th.daemon = True
    th.start()
    time.sleep(1)
    print("Flask running on http://127.0.0.1:8080")
    return th

flask_thread = start_flask_background()

# ========== 10) Cloudflared quick tunnel ==========
def ensure_cloudflared():
    exe = "cloudflared.exe" if os.name == "nt" else "cloudflared"
    if pathlib.Path(exe).exists():
        return os.path.abspath(exe)
    print("Downloading cloudflared...")
    if os.name == "nt":
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
    elif sys.platform == "darwin":
        raise RuntimeError("On macOS, run: brew install cloudflared")
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    r = requests.get(url, timeout=60); r.raise_for_status()
    with open(exe, "wb") as f: f.write(r.content)
    if os.name != "nt":
        os.chmod(exe, os.stat(exe).st_mode | stat.S_IEXEC)
    return os.path.abspath(exe)

def start_cloudflared_quick_tunnel():
    exe = ensure_cloudflared()
    cmd = [exe, "tunnel", "--url", "http://localhost:8080", "--no-autoupdate"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    public_url = None
    for _ in range(240):
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.1); continue
        print("[cloudflared]", line.strip())
        m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break
    if not public_url:
        proc.terminate()
        raise RuntimeError("Could not detect cloudflared URL")
    return public_url, proc

try:
    public_url, cloudflared_proc = start_cloudflared_quick_tunnel()
    print("\nBase:                 ", public_url)
    print("Health:               ", public_url + "/health")
    print("KB preview:           ", public_url + "/kb")
    print("Reload KB (POST):     ", public_url + "/reload_kb")
    print("Check assistant:      ", public_url + "/check-assistant")
    print("Update prompt (GET):  ", public_url + "/update_prompt")
    print("Webhook (POST):       ", public_url + "/telnyx/webhook")
    print("\nIn Telnyx: Voice → Applications → Webhook URL =", public_url + "/telnyx/webhook")
    print("Assign your DID to the SAME Voice Application, then place a real call.\n")
except Exception as e:
    print("Cloudflared start failed (non-fatal):", e)
    public_url = None
    cloudflared_proc = None

# Try local info endpoint
try:
    resp = requests.get("http://127.0.0.1:8080/update_prompt", timeout=15)
    print("Update prompt (local):", resp.status_code, resp.text[:300])
except Exception as e:
    print("Update prompt call failed (non-fatal):", e)

# ========== 11) Keep alive ==========
if __name__ == "__main__":
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        try:
            if cloudflared_proc:
                cloudflared_proc.terminate()
        except Exception:
            pass
        print("Stopped.")


=== ENV CHECK ===
TELNYX_API_KEY set?          True
TELNYX_ASSISTANT_ID set?     True
TELNYX_SIGNING_SECRET set?   False
HUMAN_FORWARD_NUMBER set?    True
APPSCRIPT_WEBHOOK_URL set?   True
APPSCRIPT_TOKEN set?         True
SHEET_TAB                    Bookings
KB_DOCX_PATH                 C:\Users\chait\Documents\AI\Voice_Agent\PlayStation_Pet_Resort_Policy_Doc.docx
KB_MAX_CHARS                 18000
KB loaded: C:\Users\chait\Documents\AI\Voice_Agent\PlayStation_Pet_Resort_Policy_Doc.docx chars: 22985
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://10.125.169.102:8080
Press CTRL+C to quit


Flask running on http://127.0.0.1:8080
[cloudflared] 2025-10-23T16:51:44Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflared] 2025-10-23T16:51:44Z INF Requesting new quick Tunnel on trycloudflare.com...


127.0.0.1 - - [23/Oct/2025 11:51:49] "GET /update_prompt HTTP/1.1" 200 -


[cloudflared] 2025-10-23T16:51:49Z INF +--------------------------------------------------------------------------------------------+
[cloudflared] 2025-10-23T16:51:49Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
[cloudflared] 2025-10-23T16:51:49Z INF |  https://factor-tue-ambassador-because.trycloudflare.com                                   |

Base:                  https://factor-tue-ambassador-because.trycloudflare.com
Health:                https://factor-tue-ambassador-because.trycloudflare.com/health
KB preview:            https://factor-tue-ambassador-because.trycloudflare.com/kb
Reload KB (POST):      https://factor-tue-ambassador-because.trycloudflare.com/reload_kb
Check assistant:       https://factor-tue-ambassador-because.trycloudflare.com/check-assistant
Update prompt (GET):   https://factor-tue-ambassador-because.trycloudflare.com/update_prompt
Webhook (POST):        https://factor-tue-ambassador-because.trycloudflar

...


== Telnyx webhook event ==
{
  "data": {
    "event_type": "call.speak.ended",
    "id": "2d558e6a-3de3-44fe-814a-1e4ef300896c",
    "occurred_at": "2025-10-23T16:56:46.977802Z",
    "payload": {
      "call_control_id": "v3:nWCndQXgUs-jagIwT_zXM-k3F7nzRMWlFSgob2BXd0eEcTD1hUX0Fw",
      "call_leg_id": "3deecf08-b031-11f0-b7e4-02420aefde1f",
      "call_session_id": "3deec6f2-b031-11f0-8949-02420aefde1f",
      "calling_party_type": "pstn",
      "client_state": null,
      "connection_id": "2793068389289952464",
      "speak_id": "-_Xw8NNaEA",
      "status": "completed"
    },
    "record_type": "event"
  },
  "meta": {
    "attempt": 1,
    "delivered_to": "https://factor-tue-ambassador-because.trycloudflare.com/telnyx/webhook"
  }
}
Event: call.speak.ended ccid: v3:nWCndQXgUs-jagIwT_zXM-k3F7nzRMWlFSgob2BXd0eEcTD1hUX0Fw from: None
GET /ai/conversations/863d046d-2c70-42ea-898f-af5595a559eb/messages => 200
{"data":[],"meta":{"total_pages":1,"total_results":0,"page_number":1,"page_size

127.0.0.1 - - [23/Oct/2025 11:58:48] "POST /telnyx/webhook HTTP/1.1" 200 -
127.0.0.1 - - [23/Oct/2025 11:58:48] "POST /telnyx/webhook HTTP/1.1" 200 -



== Telnyx webhook event ==
== Telnyx webhook event ==
{
  "data": {
    "event_type": "call.conversation.ended",
    "id": "058a431f-3a34-47be-a232-ffc94701d2de",
    "occurred_at": "2025-10-23T16:58:48.097104Z",
    "payload": {
      "assistant_id": "assistant-824811af-b3b2-4037-9bf9-a9024f11bc50",
      "call_control_id": "v3:nWCndQXgUs-jagIwT_zXM-k3F7nzRMWlFSgob2BXd0eEcTD1hUX0Fw",
      "call_leg_id": "3deecf08-b031-11f0-b7e4-02420aefde1f",
      "call_session_id": "3deec6f2-b031-11f0-8949-02420aefde1f",
      "calling_party_type": "pstn",
      "client_state": null,
      "connection_id": "2793068389289952464",
      "conversation_id": "863d046d-2c70-42ea-898f-af5595a559eb",
      "duration_sec": 126,
      "from": "+14082397947",
      "llm_model": "Qwen/Qwen3-235B-A22B",
      "stt_model": "distil-whisper/distil-large-v2",
      "to": "+18179731011",
      "tts_model_id": "NaturalHD",
      "tts_provider": "telnyx",
      "tts_voice_id": "astra"
    },
    "record_type": "event

127.0.0.1 - - [23/Oct/2025 11:59:03] "POST /telnyx/webhook HTTP/1.1" 200 -


Apps Script readLast => 200 {"ok":true}
Fallback row appended from insights.


=== ENV CHECK ===
TELNYX_API_KEY set?          True
TELNYX_ASSISTANT_ID set?     True
TELNYX_SIGNING_SECRET set?   False
HUMAN_FORWARD_NUMBER set?    True
APPSCRIPT_WEBHOOK_URL set?   True
APPSCRIPT_TOKEN set?         True
SHEET_TAB                    Bookings
KB_DOCX_PATH                 C:\Users\chait\Documents\AI\Voice_Agent\PlayStation_Pet_Resort_Policy_Doc.docx
KB_MAX_CHARS                 18000
KB loaded: C:\Users\chait\Documents\AI\Voice_Agent\PlayStation_Pet_Resort_Policy_Doc.docx chars: 22985
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://192.168.4.22:8080
Press CTRL+C to quit


Flask running on http://127.0.0.1:8080
[cloudflared] 2025-10-24T16:11:28Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflared] 2025-10-24T16:11:28Z INF Requesting new quick Tunnel on trycloudflare.com...


127.0.0.1 - - [24/Oct/2025 11:11:30] "GET /update_prompt HTTP/1.1" 404 -


[cloudflared] 2025-10-24T16:11:30Z INF +--------------------------------------------------------------------------------------------+
[cloudflared] 2025-10-24T16:11:30Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
[cloudflared] 2025-10-24T16:11:30Z INF |  https://conservation-fork-met-associates.trycloudflare.com                                |

Base:                  https://conservation-fork-met-associates.trycloudflare.com
Health:                https://conservation-fork-met-associates.trycloudflare.com/health
KB preview:            https://conservation-fork-met-associates.trycloudflare.com/kb
Reload KB (POST):      https://conservation-fork-met-associates.trycloudflare.com/reload_kb
Check assistant:       https://conservation-fork-met-associates.trycloudflare.com/check-assistant
Update prompt (GET):   https://conservation-fork-met-associates.trycloudflare.com/update_prompt
Webhook (POST):        https://conservation-fork-met-as

127.0.0.1 - - [24/Oct/2025 11:11:44] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [24/Oct/2025 11:11:44] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [24/Oct/2025 11:12:34] "GET /reload_kb HTTP/1.1" 404 -
127.0.0.1 - - [24/Oct/2025 11:12:39] "GET /reload_kb HTTP/1.1" 404 -
127.0.0.1 - - [24/Oct/2025 11:12:43] "GET /kb HTTP/1.1" 404 -


In [ ]:
#!/usr/bin/env python3
"""
PlayStayTion Voice Agent (Telnyx AI Assistant)
- Injects Knowledge Base + intake instructions dynamically via ai_assistant_start
- Polls Telnyx conversation messages mid-call to catch markers reliably
- Robust marker parsing -> Google Sheets (Apps Script) write
- Cloudflared quick tunnel
- Behavior hammer to reduce repetition + enforce markers
- Readback "nudge" if user doesn't confirm
- Insights fallback -> partial row when model forgets marker
"""

import os, json, time, hmac, hashlib, re, subprocess, sys, stat, pathlib, requests, traceback
import threading
from threading import Event
from flask import Flask, request, jsonify
from dotenv import load_dotenv
from docx import Document
from unidecode import unidecode

# ========== 0) ENV ==========
load_dotenv()

TELNYX_API_KEY        = os.getenv("TELNYX_API_KEY")
TELNYX_ASSISTANT_ID   = os.getenv("TELNYX_ASSISTANT_ID")
TELNYX_SIGNING_SECRET = os.getenv("TELNYX_SIGNING_SECRET", "")
HUMAN_FORWARD_NUMBER  = os.getenv("HUMAN_FORWARD_NUMBER")     # Telnyx/E.164 staff number

APPSCRIPT_WEBHOOK_URL = os.getenv("APPSCRIPT_WEBHOOK_URL")    # https://script.google.com/macros/s/AKfycb.../exec
APPSCRIPT_TOKEN       = os.getenv("APPSCRIPT_TOKEN", "")

SHEET_TAB             = os.getenv("GOOGLE_SHEETS_TAB_NAME", "Bookings")

BUSINESS_NAME         = os.getenv("BUSINESS_NAME", "PlayStayTion Pet Resort and Training")
BUSINESS_CITY         = os.getenv("BUSINESS_CITY", "Sadler, TX")

KB_DOCX_PATH          = os.getenv("KB_DOCX_PATH")
KB_MAX_CHARS          = int(os.getenv("KB_MAX_CHARS", "18000"))   # max KB injected per call

print("=== ENV CHECK ===")
print("TELNYX_API_KEY set?         ", bool(TELNYX_API_KEY))
print("TELNYX_ASSISTANT_ID set?    ", bool(TELNYX_ASSISTANT_ID))
print("TELNYX_SIGNING_SECRET set?  ", bool(TELNYX_SIGNING_SECRET))
print("HUMAN_FORWARD_NUMBER set?   ", bool(HUMAN_FORWARD_NUMBER))
print("APPSCRIPT_WEBHOOK_URL set?  ", bool(APPSCRIPT_WEBHOOK_URL))
print("APPSCRIPT_TOKEN set?        ", bool(APPSCRIPT_TOKEN))
print("SHEET_TAB                   ", SHEET_TAB)
print("KB_DOCX_PATH                ", KB_DOCX_PATH)
print("KB_MAX_CHARS                ", KB_MAX_CHARS)
print("======================================")

assert TELNYX_API_KEY, "Missing TELNYX_API_KEY"
assert TELNYX_ASSISTANT_ID, "Missing TELNYX_ASSISTANT_ID"
assert HUMAN_FORWARD_NUMBER, "Missing HUMAN_FORWARD_NUMBER"
assert APPSCRIPT_WEBHOOK_URL, "Missing APPSCRIPT_WEBHOOK_URL"

# ========== 1) Load KB (.docx) ==========
def load_docx_as_text(path: str) -> str:
    if not path:
        return ""
    doc = Document(path)
    chunks = []
    for p in doc.paragraphs:
        t = p.text.strip()
        if t:
            chunks.append(t)
    for tbl in doc.tables:
        for row in tbl.rows:
            cells = [c.text.strip() for c in row.cells]
            if any(cells):
                chunks.append("\t".join(cells))
    raw = "\n".join(chunks)
    raw = unidecode(raw)  # remove unusual unicode
    raw = re.sub(r"[ \t]+", " ", raw)
    raw = re.sub(r"\n{3,}", "\n\n", raw).strip()
    return raw

try:
    KB_TEXT = load_docx_as_text(KB_DOCX_PATH)
    print("KB loaded:", KB_DOCX_PATH, "chars:", len(KB_TEXT))
except Exception as e:
    KB_TEXT = ""
    print("KB load failed:", e)

# ========== 2) Telnyx helpers ==========
TELNYX_API_BASE = "https://api.telnyx.com/v2"
app = Flask(__name__)

def telnyx_post(path, payload):
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.post(url, headers={
        "Authorization": f"Bearer {TELNYX_API_KEY}",
        "Content-Type":"application/json"
    }, json=payload, timeout=25)
    print(f"POST {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    r.raise_for_status()
    return r.json()

def telnyx_get(path):
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.get(url, headers={"Authorization": f"Bearer {TELNYX_API_KEY}"}, timeout=25)
    print(f"GET {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    return r

def telnyx_get_json(path):
    """GET helper that returns JSON or {} and logs HTTP code."""
    url = f"{TELNYX_API_BASE}{path}"
    r = requests.get(url, headers={"Authorization": f"Bearer {TELNYX_API_KEY}"}, timeout=25)
    print(f"GET {path} => {r.status_code}")
    if r.text: print(r.text[:1000])
    try:
        r.raise_for_status()
        return r.json() if r.text else {}
    except Exception as e:
        print("GET error:", e)
        return {}

def verify_signature(raw, header):
    if not TELNYX_SIGNING_SECRET:
        return True
    digest = hmac.new(TELNYX_SIGNING_SECRET.encode(), raw, hashlib.sha256).hexdigest()
    return hmac.compare_digest(digest, header or "")

# ===== Conversation polling & state =====
_active_polls = {}           # { conversation_id: {"stop": Event, "last_seen_ids": set(), "ccid": str, "caller": str} }
_saved_conversations = set() # conversations where we already saved a row (marker or fallback)

# ========== 3) Prompts ==========
def build_prompt():
    kb_snippet = (KB_TEXT or "").strip()
    if len(kb_snippet) > KB_MAX_CHARS:
        kb_snippet = kb_snippet[:KB_MAX_CHARS] + "\n[KB truncated]\n"

    base = f"""
You are a warm, patient voice receptionist for {BUSINESS_NAME} in {BUSINESS_CITY}.

TOP PRIORITIES
- Be kind, calm, and easy to follow. Use short sentences. Ask ONE question at a time, then pause.
- Speak as if the caller may not be tech-savvy. Offer simple explanations and to repeat things.
- Answer only from the Knowledge Base (KB) below. If a detail is missing, say you’re not certain and offer a human transfer.
- Collect booking details conversationally (one item per turn). Read back a short summary and confirm before saving.
- Never collect or store payment information. If the caller asks to pay, or starts giving card details, stop and TRANSFER to a human immediately.

PAYMENT & PRIVACY RULES (critical)
- Do NOT ask for or record credit cards, CVV, expiration dates, bank details, or any payment numbers.
- If the caller offers payment info: say you cannot take payment over the phone, and TRANSFER to a human.
- Only collect info needed to help (name, phone, service, date/time, pet details, notes). Avoid unnecessary personal data.

IMPORTANT INSTRUCTION (HIGH PRIORITY)
- AFTER EMITTING ANY MARKER (SAVE_LEAD / TRANSFER / SEND_SMS / READY_TO_TRANSFER), IMMEDIATELY STOP SPEAKING. EMIT THE EXACT ONE-LINE MARKER AND THEN HALT — NO MORE WORDS OR AUDIO.

KNOWLEDGE BASE (authoritative; quote only from here, or say you’re not sure)
KB START
{kb_snippet}
KB END

INTAKE STYLE (strict)
- One detail per turn. Never list multiple questions at once.
- If the caller gives several details at once, briefly confirm what you captured, then ask the next missing item.
- Use gentle transitions: “Thanks. Next, what’s your dog’s name?”
- If the caller sounds unsure, explain options slowly in plain language, then ask a simple question.

FIELDS TO COLLECT (ask in this order; one per turn)
1) name (first name is fine)
2) phone (assume caller ID if not provided; read it back to confirm)
3) service (boarding | grooming | daycare | training | other)
4) date (YYYY-MM-DD or natural language like “this Friday”)
5) time (HH:MM or “morning / afternoon / evening”)
6) pet_name
7) breed
8) weight
9) notes (free text for meds, temperament, grooming style, special requests)
10) optional start_date and end_date for multi-day boarding, if relevant

READBACK BEFORE SAVING
- When you have at least name, phone, service, and a date/time (or date range), read a short summary and ask: “Is that correct?” If yes, output the marker.

OUTPUT MARKERS (exact one-line format, then stop speaking immediately)
1) SAVE_LEAD{{"name":"<first>","phone":"<caller or provided>","service":"<boarding|grooming|daycare|training|other>","date":"<YYYY-MM-DD or text>","time":"<HH:MM or text>","pet_name":"<text>","breed":"<text>","weight":"<text>","notes":"<free text>","send_sms":true|false,"start_date":"<optional>","end_date":"<optional>"}}
2) TRANSFER{{"reason":"<short reason>","priority":"normal"|"urgent"}}
3) SEND_SMS{{"message":"<short confirmation or link>","to":"<E.164 or empty to use caller>"}}
4) READY_TO_TRANSFER{{"name":"...","phone":"...","service":"...","date":"...","time":"...","pet_name":"...","breed":"...","weight":"...","notes":"..."}}

MARKER RULES
- Emit exactly one marker line and nothing after it.
- Include every key; use "" if unknown.
- After emitting a marker, stop speaking and wait.
""".strip()
    return base

def build_live_instructions():
    """Compose the exact instructions + trimmed KB we send on ai_assistant_start for THIS call."""
    core = build_prompt()
    kb_content = (KB_TEXT or "")[:KB_MAX_CHARS]
    core += (
        "\n\nKnowledge Base (verbatim, highest authority):\n----\n"
        f"{kb_content}\n"
        "----\n"
        "AFTER EMITTING ANY MARKER (SAVE_LEAD / TRANSFER / SEND_SMS / READY_TO_TRANSFER), "
        "PRINT THE EXACT ONE-LINE MARKER AND STOP. PRODUCE NO FURTHER WORDS OR AUDIO.\n"
        "\nBEHAVIOR OVERRIDES (FINAL, DO THIS EXACTLY)\n"
        "- Do not say “I didn’t catch that” more than once in the entire call. If unclear, rephrase once, then move to a simpler question or offer transfer.\n"
        "- Start intake proactively. First line after greeting: “Happy to help. What’s your first name?” Then proceed through FIELDS TO COLLECT, one item per turn.\n"
        "- When you read back the short summary and the caller says “Yes,” immediately output ONLY the one-line SAVE_LEAD{...} marker and then STOP with no additional words or audio.\n"
        "- If the caller attempts to pay or gives card numbers, output ONLY: TRANSFER{\"reason\":\"payment over phone\",\"priority\":\"urgent\"} and STOP.\n"
    )
    return core

# ========== 4) Apps Script helpers ==========
def write_row_apps_script(row: dict):
    """
    POST row to Apps Script (action=append). Accept ok:true even if rowIndex is missing.
    Optionally verify by readLast when rowIndex is absent.
    """
    payload = {
        "action": "append",
        "sheet": SHEET_TAB,
        "row": row,
        "token": APPSCRIPT_TOKEN or None
    }
    r = requests.post(APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script =>", r.status_code, r.text[:400])
    r.raise_for_status()
    resp = r.json()

    if not resp.get("ok"):
        raise RuntimeError(f"Apps Script append failed: {resp}")

    # Accept success without rowIndex
    if resp.get("rowIndex"):
        return resp

    # Optional verification
    try:
        confirm = read_last_rows_from_apps_script(n=5)
        stamp = row.get("timestamp_utc", "")
        name  = row.get("name", "")
        svc   = row.get("service", "")
        found = False
        for tail in confirm.get("rows", []):
            s = [str(c) for c in tail]
            if stamp and any(stamp in c for c in s):
                found = True; break
            if name and svc and any(name in c for c in s) and any(svc in c for c in s):
                found = True; break
        if found:
            print("Apps Script append verified via readLast (no rowIndex provided).")
            return {"ok": True, "verified_by_readLast": True}
    except Exception as e:
        print("Apps Script verification skip/error:", e)

    # Treat ok:true as success to avoid dropping leads
    return {"ok": True, "rowIndex": None, "note": "rowIndex not provided by Apps Script"}

def read_last_rows_from_apps_script(n=5):
    payload = {"action": "readLast", "sheet": SHEET_TAB, "n": n, "token": APPSCRIPT_TOKEN or None}
    r = requests.post(APPSCRIPT_WEBHOOK_URL, json=payload, timeout=20)
    print("Apps Script readLast =>", r.status_code, r.text[:400])
    r.raise_for_status()
    return r.json()

# ========== 4.5) PRIORITY SCORE (helpers; print + row fields) ==========
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Optional

# Tunable base rates (USD). Adjust as needed.
_RATE_BOARDING_PER_DAY = 85.0
_RATE_GROOMING_BASE    = 70.0
_RATE_DAYCARE_BASE     = 45.0
_RATE_TRAINING_BASE    = 120.0
_VALUE_CAP_USD         = 1000.0   # cap for normalization to 0–100

def _clamp(x, lo, hi): 
    return max(lo, min(hi, x))

def _sentiment_to_0_100(sentiment: Optional[float]):
    # sentiment in [-1,1] -> [0,100], None -> 50
    if sentiment is None: return 50
    try: s = float(sentiment)
    except: return 50
    return _clamp(round((s + 1) * 50), 0, 100)

def _urgency_from_days_left(days_left: Optional[int], max_days=14):
    if days_left is None: return 50
    if days_left <= 0: return 100
    days_left = _clamp(days_left, 0, max_days)
    return round((max_days - days_left) * 100 / max_days)

def _value_to_0_100(value_usd: Optional[float], cap_usd=_VALUE_CAP_USD):
    if value_usd is None: return 50
    try: v = float(value_usd)
    except: return 50
    return _clamp(round(min(v, cap_usd) * 100 / cap_usd), 0, 100)

@dataclass
class _Weights:
    value: float = 0.5
    urgency: float = 0.3
    sentiment: float = 0.2

def _parse_yyyy_mm_dd(s: str) -> Optional[datetime]:
    try: return datetime.strptime(s[:10], "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except: return None

def _days_until(date_str: str) -> Optional[int]:
    d = _parse_yyyy_mm_dd(date_str) if date_str else None
    if not d: return None
    now = datetime.now(timezone.utc)
    return int((d - now).days)

def _duration_days(start_date: str, end_date: str) -> int:
    s = _parse_yyyy_mm_dd(start_date) if start_date else None
    e = _parse_yyyy_mm_dd(end_date) if end_date else None
    if s and e:
        return max(1, (e - s).days)
    return 1

def _estimate_value_usd(service: str, date: str, start_date: str, end_date: str) -> float:
    svc = (service or "").strip().lower()
    if svc == "boarding":
        days = _duration_days(start_date or date, end_date or "")
        return days * _RATE_BOARDING_PER_DAY
    if svc == "grooming": return _RATE_GROOMING_BASE
    if svc == "daycare":  return _RATE_DAYCARE_BASE
    if svc == "training": return _RATE_TRAINING_BASE
    return 60.0  # fallback

def _compute_priority(value_usd: Optional[float], days_left: Optional[int], sentiment: Optional[float], w:_Weights=_Weights()):
    V = _value_to_0_100(value_usd)
    U = _urgency_from_days_left(days_left)
    S = _sentiment_to_0_100(sentiment)
    score = V*w.value + U*w.urgency + S*w.sentiment
    norm = (w.value + w.urgency + w.sentiment) or 1.0
    score = round(score / norm)
    label = "hot" if score >= 75 else "warm" if score >= 50 else "cold"
    return {"Value": V, "Urgency": U, "Sentiment": S, "Score": score, "Label": label}

def _priority_for_obj(obj: dict, conversation_id: Optional[str]=None):
    """
    Compute priority and return (score, label). Sentiment is neutral by default.
    Also prints a short summary to logs.
    """
    service = obj.get("service","") or ""
    date    = obj.get("date","") or ""
    start_d = obj.get("start_date","") or ""
    end_d   = obj.get("end_date","") or ""

    days_left = _days_until(start_d or date)
    value_usd = _estimate_value_usd(service, date, start_d, end_d)
    sentiment = None  # neutral unless you wire a sentiment source

    pr = _compute_priority(value_usd=value_usd, days_left=days_left, sentiment=sentiment)
    print(f"➡️  Priority: Score={pr['Score']} ({pr['Label']}) | Value≈${round(value_usd or 0, 2)} | DaysLeft={days_left if days_left is not None else 'n/a'} | Service={service}")
    return pr["Score"], pr["Label"]

# ========== 5) Marker scanning, nudge & poller ==========
READBACK_PATTERNS = ("Is that correct", "Does that look right", "Should I save that", "Is this okay")

def _scan_texts_for_markers(texts):
    re_save_lead = re.compile(r'SAVE_LEAD\{(.+?)\}', re.DOTALL)
    re_transfer  = re.compile(r'TRANSFER\{(.+?)\}', re.DOTALL)
    re_send_sms  = re.compile(r'SEND_SMS\{(.+?)\}', re.DOTALL)
    re_ready_xfer= re.compile(r'READY_TO_TRANSFER\{(.+?)\}', re.DOTALL)

    joined = "\n\n".join([t for t in texts if t])
    found = {"SAVE_LEAD":None,"TRANSFER":None,"SEND_SMS":None,"READY_TO_TRANSFER":None}
    if joined:
        m = re_save_lead.search(joined);  found["SAVE_LEAD"] = "{" + m.group(1).strip() + "}" if m else None
        m = re_transfer.search(joined);   found["TRANSFER"]  = "{" + m.group(1).strip() + "}" if m else None
        m = re_send_sms.search(joined);   found["SEND_SMS"]  = "{" + m.group(1).strip() + "}" if m else None
        m = re_ready_xfer.search(joined); found["READY_TO_TRANSFER"] = "{" + m.group(1).strip() + "}" if m else None
    return found

def _parse_marker_json(s):
    if not s: return {}
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(s.replace("'", '"'))
        except Exception as e:
            print("Marker JSON parse failed:", e, "| raw:", s[:400])
            return {}

def _nudge_if_readback_detected(texts, ccid):
    try:
        joined = " ".join(texts).lower()
        if any(p.lower() in joined for p in READBACK_PATTERNS):
            def _timer():
                time.sleep(15)
                try:
                    telnyx_post(f"/calls/{ccid}/actions/speak", {
                        "language":"en-US","voice":"Telnyx.NaturalHD.astra",
                        "payload":"Please say yes or no so I can save this for you."
                    })
                except Exception as e:
                    print("nudge speak err:", e)
            threading.Thread(target=_timer, daemon=True).start()
    except Exception as e:
        print("nudge check err:", e)

def _handle_found_markers(found, ccid, caller, conversation_id=None):
    global _saved_conversations
    # READY_TO_TRANSFER (legacy)
    if found.get("READY_TO_TRANSFER") and ccid:
        obj = _parse_marker_json(found["READY_TO_TRANSFER"])
        if not obj.get("phone"): obj["phone"] = caller or ""

        # >>> PRIORITY <<<
        try:
            priority_score, priority_label = _priority_for_obj(obj, conversation_id)
        except Exception as e:
            print("priority calc (READY_TO_TRANSFER) failed:", e)
            priority_score, priority_label = None, ""

        row = {
            "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "name": obj.get("name",""),
            "phone": obj.get("phone",""),
            "service": obj.get("service",""),
            "date": obj.get("date",""),
            "time": obj.get("time",""),
            "pet_name": obj.get("pet_name",""),
            "breed": obj.get("breed",""),
            "weight": obj.get("weight",""),
            "notes": obj.get("notes",""),
            "call_control_id": ccid,
            "start_date": obj.get("start_date",""),
            "end_date": obj.get("end_date",""),
            "tab": SHEET_TAB,
            "source": "legacy_ready_to_transfer",
            # NEW FIELDS:
            "priority_score": priority_score,
            "priority_label": priority_label,
        }
        try:
            write_row_apps_script(row)
            if conversation_id: _saved_conversations.add(conversation_id)
        except Exception as e:
            print("Apps Script write failed (legacy):", e)
        try:
            telnyx_post(f"/calls/{ccid}/actions/transfer", {"to": HUMAN_FORWARD_NUMBER})
            print("Transfer initiated (legacy) to", HUMAN_FORWARD_NUMBER)
        except Exception as e:
            print("Transfer err:", e)

    # SAVE_LEAD
    if found.get("SAVE_LEAD"):
        obj = _parse_marker_json(found["SAVE_LEAD"])

        # >>> PRIORITY <<<
        try:
            priority_score, priority_label = _priority_for_obj(obj, conversation_id)
        except Exception as e:
            print("priority calc (SAVE_LEAD) failed:", e)
            priority_score, priority_label = None, ""

        row = {
            "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "name": obj.get("name",""),
            "phone": obj.get("phone") or caller or "",
            "service": obj.get("service",""),
            "date": obj.get("date",""),
            "time": obj.get("time",""),
            "pet_name": obj.get("pet_name",""),
            "breed": obj.get("breed",""),
            "weight": obj.get("weight",""),
            "notes": obj.get("notes",""),
            "call_control_id": ccid or "",
            "start_date": obj.get("start_date",""),
            "end_date": obj.get("end_date",""),
            "tab": SHEET_TAB,
            "source": "save_lead",
            # NEW FIELDS:
            "priority_score": priority_score,
            "priority_label": priority_label,
        }
        try:
            write_row_apps_script(row)
            print("Lead saved (poller/webhook)")
            if conversation_id: _saved_conversations.add(conversation_id)
        except Exception as e:
            print("Apps Script write failed:", e)

        if obj.get("send_sms") and (obj.get("phone") or caller):
            try:
                telnyx_post("/messages", {
                    "from": HUMAN_FORWARD_NUMBER,
                    "to": obj.get("phone") or caller,
                    "text": "Thanks! We saved your request and will follow up shortly."
                })
                print("Confirmation SMS sent")
            except Exception as e:
                print("SMS err:", e)

    # TRANSFER
    if found.get("TRANSFER") and ccid:
        obj = _parse_marker_json(found["TRANSFER"])
        try:
            telnyx_post(f"/calls/{ccid}/actions/transfer", {"to": HUMAN_FORWARD_NUMBER})
            print("Transfer initiated:", obj.get("reason",""))
        except Exception as e:
            print("Transfer err:", e)

    # SEND_SMS
    if found.get("SEND_SMS"):
        obj = _parse_marker_json(found["SEND_SMS"])
        to = obj.get("to") or caller
        msg = (obj.get("message") or "").strip()
        if to and msg:
            try:
                telnyx_post("/messages", {"from": HUMAN_FORWARD_NUMBER, "to": to, "text": msg})
                print("SMS sent to", to)
            except Exception as e:
                print("SMS err:", e)
        else:
            print("SEND_SMS missing to/message")

def _poll_conversation_messages(conversation_id, ccid, caller):
    """Fetch assistant messages periodically and look for markers."""
    print(f"[poller] start for conversation {conversation_id}")
    stop_ev = _active_polls[conversation_id]["stop"]
    last_seen = _active_polls[conversation_id]["last_seen_ids"]

    try:
        while not stop_ev.is_set():
            data = telnyx_get_json(f"/ai/conversations/{conversation_id}/messages")
            msgs = (data.get("data") or []) if isinstance(data, dict) else []
            texts = []
            new_ids = 0
            for m in msgs:
                mid = m.get("id")
                if mid and mid in last_seen:
                    continue
                if mid:
                    last_seen.add(mid)
                    new_ids += 1
                t = m.get("text") or m.get("content") or m.get("message") or ""
                if isinstance(t, str) and t.strip():
                    texts.append(t)

            if new_ids:
                print(f"[poller] {new_ids} new message(s) for {conversation_id}")
                if texts:
                    print("[poller] sample:", (texts[-1][:300].replace("\n"," ")))
                _nudge_if_readback_detected(texts, ccid)
                found = _scan_texts_for_markers(texts)
                if any(found.values()):
                    print("[poller] marker(s) detected:", [k for k,v in found.items() if v])
                    _handle_found_markers(found, ccid, caller, conversation_id)

            stop_ev.wait(1.8)
    finally:
        print(f"[poller] stop for conversation {conversation_id}")

# ========== 6) Insights fallback ==========
def _extract_fields_from_insight(text: str, caller: str):
    """Very light heuristic extraction for fallback rows."""
    service = ""
    for kw in ("boarding","grooming","daycare","training"):
        if kw in text.lower():
            service = kw; break

    # naive phone digits if mentioned, else caller
    phone_match = re.search(r'(?:\+?\d[\d\-\s]{7,}\d)', text)
    phone = re.sub(r'\D','', phone_match.group(0)) if phone_match else (caller or "")
    if phone and not phone.startswith("+") and len(phone) >= 10:
        phone = "+" + phone[-10:]

    # try pet name (capitalized word before "weigh"/"weight"/"dog"/"cat")
    pet_name = ""
    m = re.search(r'([A-Z][a-z]{1,20})[^.!?]{0,30}\b(?:dog|cat|pet|weigh|weight)', text)
    if m: pet_name = m.group(1)

    # try weight (first number + lb/lbs/pounds)
    weight = ""
    m = re.search(r'(\d{1,3})(?:\s?(?:lb|lbs|pounds)?)', text.lower())
    if m: weight = m.group(1)

    name = ""
    # cheap guess: first capitalized word at sentence start not matching pet name
    m = re.search(r'([A-Z][a-z]{1,20})\b', text)
    if m and m.group(1) != pet_name:
        name = m.group(1)

    return {
        "name": name,
        "phone": phone,
        "service": service or "other",
        "pet_name": pet_name,
        "weight": weight
    }

def _insights_fallback_append(conversation_id, ccid, caller, text):
    """Append a partial row if no marker was captured for this conversation."""
    if conversation_id in _saved_conversations:
        return  # already saved by marker
    fields = _extract_fields_from_insight(text or "", caller)

    # Priority for fallback (rough, often missing dates → urgency neutral)
    try:
        pr_score, pr_label = _priority_for_obj({
            "service": fields.get("service",""),
            "date": "",
            "start_date": "",
            "end_date": ""
        }, conversation_id)
    except Exception as e:
        print("priority calc (insights_fallback) failed:", e)
        pr_score, pr_label = None, ""

    row = {
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "name": fields.get("name",""),
        "phone": fields.get("phone") or caller or "",
        "service": fields.get("service","other"),
        "date": "",
        "time": "",
        "pet_name": fields.get("pet_name",""),
        "breed": "",
        "weight": fields.get("weight",""),
        "notes": f"[insights_fallback] {text[:180]}",
        "call_control_id": ccid or "",
        "start_date": "",
        "end_date": "",
        "tab": SHEET_TAB,
        "source": "insights_fallback",
        "nonce": f"insights-{int(time.time())}",
        # NEW FIELDS:
        "priority_score": pr_score,
        "priority_label": pr_label,
    }
    try:
        write_row_apps_script(row)
        _saved_conversations.add(conversation_id)
        print("Fallback row appended from insights.")
    except Exception as e:
        print("Apps Script write failed (insights fallback):", e)

# ========== 7) Flask routes ==========
@app.get("/health")
def health():
    return jsonify({"ok": True})

@app.get("/kb")
def kb_preview():
    return jsonify({"ok": True, "length": len(KB_TEXT), "snippet_first_800": (KB_TEXT or "")[:800]})

@app.post("/reload_kb")
def reload_kb():
    global KB_TEXT
    path = request.args.get("path") or KB_DOCX_PATH
    try:
        KB_TEXT = load_docx_as_text(path)
        return jsonify({"ok": True, "path": path, "length": len(KB_TEXT)})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/check-assistant")
def check_assistant():
    r = telnyx_get(f"/ai/assistants/{TELNYX_ASSISTANT_ID}")
    return (r.text, r.status_code, {"Content-Type":"application/json"})

# We don't patch the portal assistant; we inject live per call.
@app.get("/update_prompt")
def update_prompt():
    return jsonify({
        "ok": True,
        "note": "Skipping portal update. Instructions are injected dynamically via ai_assistant_start for each call.",
        "kb_len": len((KB_TEXT or "")[:KB_MAX_CHARS])
    })

@app.get("/gs/health")
def gs_health():
    nonce = f"probe-{int(time.time())}"
    row = {
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "name": "HealthCheck",
        "phone": "",
        "service": "healthcheck",
        "date": "",
        "time": "",
        "pet_name": "",
        "breed": "",
        "weight": "",
        "notes": "gs_health probe",
        "call_control_id": "",
        "start_date": "",
        "end_date": "",
        "source": "healthcheck",
        "nonce": nonce,
        # Optional: neutral priority for health row
        "priority_score": 50,
        "priority_label": "warm",
    }
    try:
        resp = write_row_apps_script(row)
        return jsonify({"ok": True, "append_response": resp, "nonce": nonce})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

@app.get("/gs/check_last")
def gs_check_last():
    n = int(request.args.get("n", "5"))
    nonce = request.args.get("nonce", "")
    try:
        data = read_last_rows_from_apps_script(n=n)
        found = False
        for row in data.get("rows", []):
            if nonce and nonce in [str(c) for c in row]:
                found = True
                break
        return jsonify({"ok": True, "found_nonce": found, "nonce": nonce, "raw": data})
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

# ========== 8) Telnyx webhook ==========
@app.post("/telnyx/webhook")
def telnyx_webhook():
    raw_body = request.get_data()
    sig = request.headers.get("Telnyx-Signature")
    if not verify_signature(raw_body, sig):
        return jsonify({"error": "bad signature"}), 400

    evt = request.get_json(silent=True) or {}
    data    = (evt.get("data") or {})
    payload = (data.get("payload") or {})
    etype   = data.get("event_type")
    ccid    = payload.get("call_control_id")
    caller  = payload.get("from")
    conv_id = (payload or {}).get("conversation_id")

    # Verbose log
    print("\n== Telnyx webhook event ==")
    try:
        print(json.dumps(evt, indent=2)[:4000])
    except Exception:
        print(str(evt)[:2000])
    print("Event:", etype, "ccid:", ccid, "from:", caller)

    # Greeting
    if etype == "call.initiated" and ccid:
        try:
            telnyx_post(f"/calls/{ccid}/actions/answer", {})
            telnyx_post(f"/calls/{ccid}/actions/speak", {
                "language":"en-US","voice":"Telnyx.NaturalHD.astra",
                "payload": f"Hi, thanks for calling {BUSINESS_NAME}. One moment please."
            })
        except Exception as e:
            print("answer/speak err:", e); traceback.print_exc()

    # Start assistant with live instructions
    if etype == "call.answered" and ccid:
        try:
            live_instructions = build_live_instructions()
            resp = telnyx_post(f"/calls/{ccid}/actions/ai_assistant_start", {
                "assistant": {
                    "id": TELNYX_ASSISTANT_ID,
                    "instructions": live_instructions
                }
            })
            print("ai_assistant_start response:", resp)
        except Exception as e:
            print("ai_assistant_start err:", e); traceback.print_exc()

    # When conversation is created, start poller
    if etype == "call.conversation.created":
        conv_id = (payload or {}).get("conversation_id")
        if conv_id and ccid and conv_id not in _active_polls:
            stop_ev = Event()
            _active_polls[conv_id] = {"stop": stop_ev, "last_seen_ids": set(), "ccid": ccid, "caller": payload.get("from")}
            threading.Thread(
                target=_poll_conversation_messages,
                args=(conv_id, ccid, payload.get("from")),
                daemon=True
            ).start()

    # Mid-call: parse any message-like webhooks (best-effort)
    if etype in (
        "call.conversation_insights.generated",
        "call.conversation.updated",
        "call.message.created",
        "ai_assistant.message.created",
        "ai_assistant.conversation.updated",
    ):
        texts = []
        for res in (payload.get("results") or []):
            if isinstance(res, dict) and isinstance(res.get("result"), str):
                texts.append(res["result"])
        for fld in ("transcript","text","message","content"):
            val = payload.get(fld)
            if isinstance(val, str):
                texts.append(val)
        for msg in (payload.get("messages") or []):
            if isinstance(msg, dict):
                for fld in ("text","content","message"):
                    if isinstance(msg.get(fld), str):
                        texts.append(msg[fld])

        if texts:
            print("Scanned last outputs (most recent 10):")
            for t in texts[-10:]:
                print("  •", (t[:250].replace("\n", " ")))

            _nudge_if_readback_detected(texts, ccid)
            found = _scan_texts_for_markers(texts)
            if any(found.values()):
                print("[webhook] marker(s) detected:", [k for k,v in found.items() if v])
                _handle_found_markers(found, ccid, caller, conv_id)

        # Insights fallback (when present)
        if etype == "call.conversation_insights.generated":
            try:
                insight_texts = [r.get("result","") for r in (payload.get("results") or []) if isinstance(r, dict)]
                insight_blob = " ".join(insight_texts).strip()
                if insight_blob and conv_id and conv_id not in _saved_conversations:
                    _insights_fallback_append(conv_id, ccid, caller, insight_blob)
            except Exception as e:
                print("insights fallback err:", e)

    # Stop poller on end/hangup
    if etype in ("call.conversation.ended", "call.hangup"):
        conv_id = (payload or {}).get("conversation_id")
        if conv_id and conv_id in _active_polls:
            _active_polls[conv_id]["stop"].set()
            try: del _active_polls[conv_id]
            except Exception: pass

    return jsonify({"ok": True})

# ========== 9) Launch Flask (Jupyter-safe) ==========
def start_flask_background():
    th = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8080, debug=False, use_reloader=False))
    th.daemon = True
    th.start()
    time.sleep(1)
    print("Flask running on http://127.0.0.1:8080")
    return th

flask_thread = start_flask_background()

# ========== 10) Cloudflared quick tunnel ==========
def ensure_cloudflared():
    exe = "cloudflared.exe" if os.name == "nt" else "cloudflared"
    if pathlib.Path(exe).exists():
        return os.path.abspath(exe)
    print("Downloading cloudflared...")
    if os.name == "nt":
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
    elif sys.platform == "darwin":
        raise RuntimeError("On macOS, run: brew install cloudflared")
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    r = requests.get(url, timeout=60); r.raise_for_status()
    with open(exe, "wb") as f: f.write(r.content)
    if os.name != "nt":
        os.chmod(exe, os.stat(exe).st_mode | stat.S_IEXEC)
    return os.path.abspath(exe)

def start_cloudflared_quick_tunnel():
    exe = ensure_cloudflared()
    cmd = [exe, "tunnel", "--url", "http://localhost:8080", "--no-autoupdate"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    public_url = None
    for _ in range(240):
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.1); continue
        print("[cloudflared]", line.strip())
        m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break
    if not public_url:
        proc.terminate()
        raise RuntimeError("Could not detect cloudflared URL")
    return public_url, proc

try:
    public_url, cloudflared_proc = start_cloudflared_quick_tunnel()
    print("\nBase:                 ", public_url)
    print("Health:               ", public_url + "/health")
    print("KB preview:           ", public_url + "/kb")
    print("Reload KB (POST):     ", public_url + "/reload_kb")
    print("Check assistant:      ", public_url + "/check-assistant")
    print("Update prompt (GET):  ", public_url + "/update_prompt")
    print("Webhook (POST):       ", public_url + "/telnyx/webhook")
    print("\nIn Telnyx: Voice → Applications → Webhook URL =", public_url + "/telnyx/webhook")
    print("Assign your DID to the SAME Voice Application, then place a real call.\n")
except Exception as e:
    print("Cloudflared start failed (non-fatal):", e)
    public_url = None
    cloudflared_proc = None

# Try local info endpoint
try:
    resp = requests.get("http://127.0.0.1:8080/update_prompt", timeout=15)
    print("Update prompt (local):", resp.status_code, resp.text[:300])
except Exception as e:
    print("Update prompt call failed (non-fatal):", e)

# ========== 11) Keep alive ==========
if __name__ == "__main__":
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        try:
            if cloudflared_proc:
                cloudflared_proc.terminate()
        except Exception:
            pass
        print("Stopped.")


=== ENV CHECK ===
TELNYX_API_KEY set?          True
TELNYX_ASSISTANT_ID set?     True
TELNYX_SIGNING_SECRET set?   False
HUMAN_FORWARD_NUMBER set?    True
APPSCRIPT_WEBHOOK_URL set?   True
APPSCRIPT_TOKEN set?         True
SHEET_TAB                    Bookings
KB_DOCX_PATH                 C:\Users\chait\Documents\AI\Voice_Agent\PlayStation_Pet_Resort_Policy_Doc.docx
KB_MAX_CHARS                 18000
KB loaded: C:\Users\chait\Documents\AI\Voice_Agent\PlayStation_Pet_Resort_Policy_Doc.docx chars: 22985
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://192.168.4.22:8080
Press CTRL+C to quit


Flask running on http://127.0.0.1:8080
[cloudflared] 2025-10-24T17:25:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflared] 2025-10-24T17:25:36Z INF Requesting new quick Tunnel on trycloudflare.com...
[cloudflared] 2025-10-24T17:25:40Z INF +--------------------------------------------------------------------------------------------+
[cloudflared] 2025-10-24T17:25:40Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be r